In [1]:
# ============================================
# CELLULE 1 — Imports et configuration
# ============================================
import pandas as pd
import numpy as np
import zipfile
import warnings
warnings.filterwarnings('ignore')

PATH_BEDS     = r'C:\Users\hp\healthsim-simulation\data\Beds-Open-Overnight-Web_File-Q4-2023-24-Final.xlsx'
PATH_DOCTORS  = r'C:\Users\hp\healthsim-simulation\data\NHS Workforce Statistics, March 2024 Doctors by Grade and Specialty.xlsx'
PATH_NURSES   = r'C:\Users\hp\healthsim-simulation\data\NHS Workforce Statistics, March 2024 Staff Group, Care Setting and Level (1).xlsx'
PATH_SHMI_ZIP = r'C:\Users\hp\healthsim-simulation\data\SHMI data, Apr23-Mar24.zip'
PATH_CDI      = r'C:\Users\hp\healthsim-simulation\data\Table_6_CDI_april_2023_to_march_2024-os.ods'
PATH_PROV     = r'C:\Users\hp\healthsim-simulation\data\hosp-epis-stat-admi-hosp-prov-2023-24-tab.xlsx'
PATH_ERIC     = r'C:\Users\hp\healthsim-simulation\data\ERIC - 2023_24 - Trust data.csv'
PATH_NIDC     = r'C:\Users\hp\healthsim-simulation\data\National-Imaging-Data-Collection-Asset-Count-2023-24-v1.xlsx'
PATH_ORG      = r'C:\Users\hp\healthsim-simulation\data\NHS Workforce Statistics, March 2024 England and Organisation.xlsx'

print("✅ Imports OK")
print("✅ Chemins configurés")

C:\Users\hp\AppData\Roaming\Python\Python312\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\hp\AppData\Roaming\Python\Python312\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


✅ Imports OK
✅ Chemins configurés


In [2]:
# CELLULE 2 CORRIGÉE — Vérifier les colonnes
df_beds_raw = pd.read_excel(PATH_BEDS, sheet_name='NHS Trust by Sector', header=14)
print("Colonnes disponibles :")
for col in df_beds_raw.columns:
    print(f"  - '{col}'")

Colonnes disponibles :
  - 'Unnamed: 0'
  - 'Year'
  - 'Period'
  - 'Region Code'
  - 'Org Code'
  - 'Org Name'
  - 'Total '
  - 'General & Acute'
  - 'Learning Disabilities'
  - 'Maternity'
  - 'Mental Illness'
  - 'Unnamed: 11'
  - 'Total .1'
  - 'General & Acute.1'
  - 'Learning Disabilities.1'
  - 'Maternity.1'
  - 'Mental Illness.1'
  - 'Unnamed: 17'
  - 'Total .2'
  - 'General & Acute.2'
  - 'Learning Disabilities.2'
  - 'Maternity.2'
  - 'Mental Illness.2'


In [3]:
pip install --upgrade openpyxl


In [4]:
# CELLULE 2 — Chargement NHS Beds (corrigé)
df_beds_raw = pd.read_excel(PATH_BEDS, sheet_name='NHS Trust by Sector', header=14)

df_beds = df_beds_raw[['Org Code', 'Org Name',
                        'Total ', 'Total .1',
                        'Total .2']].copy()

df_beds.columns = ['Trust_code', 'Trust_name',
                   'lits_disponibles', 'lits_occupes',
                   'C2_occupation_pct']

# Garder seulement les vrais codes trust (3 caractères)
df_beds = df_beds[df_beds['Trust_code'].astype(str).str.len() == 3]

# Convertir en numérique
df_beds['lits_disponibles']  = pd.to_numeric(df_beds['lits_disponibles'], errors='coerce')
df_beds['lits_occupes']      = pd.to_numeric(df_beds['lits_occupes'], errors='coerce')
df_beds['C2_occupation_pct'] = pd.to_numeric(df_beds['C2_occupation_pct'], errors='coerce')

# Convertir en pourcentage (0.94 → 94%)
df_beds['C2_occupation_pct'] = df_beds['C2_occupation_pct'] * 100

# Calculer A8
df_beds['A8_lits_vacants_pct'] = 100 - df_beds['C2_occupation_pct']

# Supprimer lignes manquantes
df_beds = df_beds.dropna(subset=['lits_disponibles', 'C2_occupation_pct'])

print(f"✅ NHS Beds chargé : {len(df_beds)} trusts")
print(f"\nAperçu :")
print(df_beds.head(3).to_string())
print(f"\nStats C2 et A8 :")
print(df_beds[['C2_occupation_pct', 'A8_lits_vacants_pct']].describe())

✅ NHS Beds chargé : 185 trusts

Aperçu :
  Trust_code                                         Trust_name  lits_disponibles  lits_occupes  C2_occupation_pct  A8_lits_vacants_pct
2        R1H                             BARTS HEALTH NHS TRUST       2072.923077   1967.813187          94.929388             5.070612
3        R1K  LONDON NORTH WEST UNIVERSITY HEALTHCARE NHS TRUST       1089.978022   1021.153846          93.685728             6.314272
4        RAL             ROYAL FREE LONDON NHS FOUNDATION TRUST       1203.109890    922.637363          76.687705            23.312295

Stats C2 et A8 :
       C2_occupation_pct  A8_lits_vacants_pct
count         185.000000           185.000000
mean           88.559857            11.440143
std             8.028079             8.028079
min            50.240427             0.826781
25%            86.735088             6.466237
50%            90.265487             9.734513
75%            93.533763            13.264912
max            99.173219     

In [5]:
# CELLULE 3 — Vérifier les colonnes SHMI
import zipfile

with zipfile.ZipFile(PATH_SHMI_ZIP, 'r') as z:
    z.extractall(r'C:\Users\hp\healthsim-simulation\data\SHMI')

shmi_file = r'C:\Users\hp\healthsim-simulation\data\SHMI\SHMI data\SHMI_data_at_trust_level_Apr23-Mar24_csv.csv'
df_shmi = pd.read_csv(shmi_file)

print("Colonnes disponibles :")
for col in df_shmi.columns:
    print(f"  - '{col}'")

print("\nPremières lignes :")
print(df_shmi.head(3).to_string())

Colonnes disponibles :
  - 'INDICATOR_CODE'
  - 'PROVIDER_CODE'
  - 'PROVIDER_NAME'
  - 'SHMI_VALUE'
  - 'SHMI_BANDING'
  - 'SPELLS'
  - 'OBSERVED'
  - 'EXPECTED'
  - 'PO_LL'
  - 'PO_UL'
  - 'OD_LL'
  - 'OD_UL'

Premières lignes :
  INDICATOR_CODE PROVIDER_CODE                                                   PROVIDER_NAME SHMI_VALUE SHMI_BANDING  SPELLS OBSERVED EXPECTED   PO_LL   PO_UL   OD_LL   OD_UL
0         I00699           RCF                                   AIREDALE NHS FOUNDATION TRUST     0.9745            2   32270     1095     1125  0.9104  1.0956  0.8848  1.1302
1         I00699           RTK           ASHFORD AND ST PETER'S HOSPITALS NHS FOUNDATION TRUST     0.9874            2   58460     1635     1660  0.9258  1.0782  0.8888  1.1251
2         I00699           RF4  BARKING, HAVERING AND REDBRIDGE UNIVERSITY HOSPITALS NHS TRUST     0.8984            2  115830     2645     2945   0.944  1.0583  0.8927  1.1202


In [6]:
# ============================================
# CELLULE 3 — Chargement NHS SHMI (corrigé)
# ============================================
shmi_file = r'C:\Users\hp\healthsim-simulation\data\SHMI data, Apr23-Mar24.zip'
import zipfile, os

with zipfile.ZipFile(shmi_file, 'r') as z:
    z.extractall(r'C:\Users\hp\healthsim-simulation\data\SHMI')

shmi_csv = r'C:\Users\hp\healthsim-simulation\data\SHMI\SHMI data\SHMI_data_at_trust_level_Apr23-Mar24_csv.csv'
df_shmi = pd.read_csv(shmi_csv)

df_shmi = df_shmi[['PROVIDER_CODE', 'PROVIDER_NAME',
                    'SHMI_VALUE', 'SHMI_BANDING',
                    'OBSERVED', 'EXPECTED',
                    'SPELLS']].copy()

df_shmi.columns = ['Trust_code', 'Trust_name_shmi',
                   'SHMI_VALUE', 'SHMI_banding',
                   'deces_observes', 'deces_attendus',
                   'spells_total']

# Convertir en numérique
for col in ['SHMI_VALUE', 'deces_observes', 'deces_attendus', 'spells_total']:
    df_shmi[col] = pd.to_numeric(df_shmi[col], errors='coerce')

# ✅ C3 en pourcentage — compatible HealthSim
df_shmi['C3_mortalite_pct'] = (df_shmi['deces_observes'] / 
                                df_shmi['spells_total'] * 100)

print(f"✅ NHS SHMI chargé : {len(df_shmi)} trusts")
print(f"\nAperçu :")
print(df_shmi[['Trust_code', 'C3_mortalite_pct', 'spells_total']].head(3).to_string())
print(f"\nStats C3 % :")
print(df_shmi['C3_mortalite_pct'].describe())

✅ NHS SHMI chargé : 119 trusts

Aperçu :
  Trust_code  C3_mortalite_pct  spells_total
0        RCF          3.393244       32270.0
1        RTK          2.796784       58460.0
2        RF4          2.283519      115830.0

Stats C3 % :
count    118.000000
mean       3.283792
std        0.737346
min        1.260902
25%        2.839210
50%        3.321513
75%        3.754134
max        5.408766
Name: C3_mortalite_pct, dtype: float64


In [7]:
# ============================================
# CELLULE 4 — Chargement CDI Infections (corrigé)
# ============================================
df_cdi_raw = pd.read_excel(PATH_CDI, engine='odf',
                            sheet_name='data', header=2)

df_cdi_raw.columns = ['Collection', 'Org_type', 'Org_code', 'Org_name',
                       'UKHSA', 'ICB', 'Region', 'Year', 'Month',
                       'Metric', 'Figure', 'Signed', 'Changed']

# Filtrer NHS acute trust + Total cases
df_cdi_filtered = df_cdi_raw[
    (df_cdi_raw['Org_type'] == 'NHS acute trust') &
    (df_cdi_raw['Metric'] == 'Total cases')
].copy()

df_cdi_filtered['Figure'] = pd.to_numeric(df_cdi_filtered['Figure'], errors='coerce')

# Total annuel par trust
df_cdi = df_cdi_filtered.groupby(['Org_code', 'Org_name'])['Figure'].sum().reset_index()
df_cdi.columns = ['Trust_code', 'Trust_name_cdi', 'CDI_total_cases']

# ✅ C4 en % — fusion avec spells_total depuis SHMI
df_cdi = df_cdi.merge(df_shmi[['Trust_code', 'spells_total']], 
                       on='Trust_code', how='left')

df_cdi['C4_infections_pct'] = (df_cdi['CDI_total_cases'] / 
                                df_cdi['spells_total'] * 100)

print(f"✅ NHS CDI chargé : {len(df_cdi)} trusts")
print(f"\nAperçu :")
print(df_cdi[['Trust_code', 'CDI_total_cases', 
              'spells_total', 'C4_infections_pct']].head(3).to_string())
print(f"\nStats C4 % :")
print(df_cdi['C4_infections_pct'].describe())

✅ NHS CDI chargé : 136 trusts

Aperçu :
  Trust_code  CDI_total_cases  spells_total  C4_infections_pct
0        R0A              326      188940.0           0.172542
1        R0B              111       80960.0           0.137105
2        R0D              184       97940.0           0.187870

Stats C4 % :
count    118.000000
mean       0.185913
std        0.068093
min        0.042909
25%        0.140500
50%        0.177353
75%        0.229690
max        0.421355
Name: C4_infections_pct, dtype: float64


In [8]:
pip install odfpy


In [9]:
# ============================================
# CELLULE 5 — Chargement Hospital Providers
# ============================================
df_prov_raw = pd.read_excel(PATH_PROV, 
                             sheet_name='Hospital Providers', 
                             header=None)

# Trouver la ligne header
for i in range(30):
    row = df_prov_raw.iloc[i]
    vals = [str(v) for v in row if str(v) != 'nan']
    if len(vals) > 5:
        header_row = i
        break

print(f"Header trouvé à la ligne : {header_row}")

# Recharger avec le bon header
df_prov = pd.read_excel(PATH_PROV, 
                         sheet_name='Hospital Providers', 
                         header=header_row)

print("\nColonnes disponibles :")
for col in df_prov.columns:
    print(f"  - '{col}'")

Header trouvé à la ligne : 16

Colonnes disponibles :
  - 'Hospital provider code and description†'
  - 'Unnamed: 1'
  - 'Unnamed: 2'
  - 'Unnamed: 3'
  - 'Unnamed: 4'
  - 'Unnamed: 5'
  - 'Unnamed: 6'
  - 'Finished consultant episodes'
  - 'Finished Admission Episodes'
  - 'Male 
(FCE)'
  - 'Female 
(FCE)'
  - 'Gender Unknown 
(FCE)'
  - 'Emergency 
(FAE)'
  - 'Waiting list 
(FAE)'
  - 'Planned (FAE)'
  - 'Other (FAE)'
  - 'Mean time waited 
(Days)'
  - 'Median time waited 
(Days)'
  - 'Mean length of stay 
(Days)'
  - 'Median length of stay 
(Days)'
  - 'Mean age 
(Years)'
  - 'Age 0 
(FCE)'
  - 'Age 1-4 
(FCE)'
  - 'Age 5-9 
(FCE)'
  - 'Age 10-14 
(FCE)'
  - 'Age 15 
(FCE)'
  - 'Age 16 
(FCE)'
  - 'Age 17 
(FCE)'
  - 'Age 18 
(FCE)'
  - 'Age 19 
(FCE)'
  - 'Age 20-24 
(FCE)'
  - 'Age 25-29 
(FCE)'
  - 'Age 30-34 
(FCE)'
  - 'Age 35-39 
(FCE)'
  - 'Age 40-44 
(FCE)'
  - 'Age 45-49 
(FCE)'
  - 'Age 50-54 
(FCE)'
  - 'Age 55-59 
(FCE)'
  - 'Age 60-64 
(FCE)'
  - 'Age 65-69 
(FCE)'
  - 

In [10]:
# ============================================
# CELLULE 5 — Chargement Hospital Providers (corrigé)
# ============================================
df_prov_raw = pd.read_excel(PATH_PROV,
                             sheet_name='Hospital Providers',
                             header=16)

# Garder colonnes utiles
df_prov = df_prov_raw[['Hospital provider code and description†',
                        'Unnamed: 1',
                        'Finished Admission Episodes',
                        'Emergency \n(FAE)',
                        'Mean length of stay \n(Days)',
                        'FCE bed days',
                        'Emergency \n(FAE).1',
                        'Elective\n(FAE)',
                        'Other\n(FAE)']].copy()

# Renommer
df_prov.columns = ['Trust_code', 'Trust_name_prov',
                   'total_admissions',
                   'emergency_fae_admission',
                   'A2_dms_jours',
                   'fce_bed_days',
                   'emergency_fae_discharge',
                   'elective_fae_discharge',
                   'other_fae_discharge']

# Garder seulement codes trust 3 caractères
df_prov = df_prov[df_prov['Trust_code'].astype(str).str.len() == 3]

# Convertir en numérique
for col in ['total_admissions', 'emergency_fae_admission',
            'A2_dms_jours', 'fce_bed_days',
            'emergency_fae_discharge', 'elective_fae_discharge',
            'other_fae_discharge']:
    df_prov[col] = pd.to_numeric(df_prov[col], errors='coerce')

# ✅ A9 corrigé — Other FAE discharge / Total × 100
df_prov['A9_taux_transfert_pct'] = (df_prov['other_fae_discharge'] /
                                     df_prov['total_admissions'] * 100)

# ✅ pct admissions urgentes
df_prov['pct_admissions_urgentes'] = (df_prov['emergency_fae_admission'] /
                                       df_prov['total_admissions'] * 100)

# Supprimer lignes manquantes
df_prov = df_prov.dropna(subset=['A2_dms_jours', 'A9_taux_transfert_pct'])

print(f"✅ Hospital Providers chargé : {len(df_prov)} trusts")
print(f"\nAperçu :")
print(df_prov[['Trust_code', 'A2_dms_jours', 
               'A9_taux_transfert_pct',
               'pct_admissions_urgentes']].head(3).to_string())
print(f"\nStats :")
print(df_prov[['A2_dms_jours', 'A9_taux_transfert_pct']].describe())

✅ Hospital Providers chargé : 150 trusts

Aperçu :
  Trust_code  A2_dms_jours  A9_taux_transfert_pct  pct_admissions_urgentes
3        Y56      5.083816               4.601895                31.061051
4        Y58      4.699411               3.137688                38.904319
5        Y59      5.771161               2.467638                40.296955

Stats :
       A2_dms_jours  A9_taux_transfert_pct
count    150.000000             150.000000
mean      16.788306               3.080022
std       45.277442               4.215239
min        0.347465               0.000000
25%        3.924706               0.795248
50%        4.482360               1.456726
75%        5.849198               4.598963
max      363.739344              38.692027


In [11]:
# ============================================
# CELLULE 6 — Médecins et Infirmiers par hôpital
# ============================================
PATH_ORG = r'C:\Users\hp\healthsim-simulation\data\NHS Workforce Statistics, March 2024 England and Organisation.xlsx'

df_org_raw = pd.read_excel(PATH_ORG, 
                            sheet_name='3. NHSE, Org & SG - FTE', 
                            header=5)

print("Colonnes disponibles :")
for col in df_org_raw.columns:
    print(f"  - '{col}'")

Colonnes disponibles :
  - 'NHS England region code'
  - 'NHS England region name'
  - 'ICS code'
  - 'ICS name'
  - 'Organisation name'
  - 'Organisation code'
  - 'Total'
  - 'Professionally qualified clinical staff'
  - 'HCHS Doctors'
  - 'Consultant'
  - 'Associate Specialist'
  - 'Specialty Doctor'
  - 'Staff Grade'
  - 'Specialty Registrar'
  - 'Core Training'
  - 'Foundation Doctor Year 2'
  - 'Foundation Doctor Year 1'
  - 'Hospital Practitioner / Clinical Assistant'
  - 'Other and Local HCHS Doctor Grades'
  - 'Nurses & health visitors'
  - 'Midwives'
  - 'Ambulance staff'
  - 'Scientific, therapeutic & technical staff'
  - 'Support to clinical staff'
  - 'Support to doctors, nurses & midwives'
  - 'Support to ambulance staff'
  - 'Support to ST&T staff'
  - 'NHS infrastructure support'
  - 'Central functions'
  - 'Hotel, property & estates'
  - 'Senior managers'
  - 'Managers'
  - 'Other staff or those with unknown classification'


In [12]:
# ============================================
# CELLULE 5B — Chargement A4 depuis AEQI
# ============================================
import requests
import io

url = "https://files.digital.nhs.uk/23/C1A9E0/aeqi_open_data_2024_03.csv"
print("Téléchargement AEQI...")
response = requests.get(url, timeout=120)
df_aeqi = pd.read_csv(io.StringIO(response.content.decode('utf-8')))
print(f"✅ AEQI chargé : {df_aeqi.shape}")

# Filtrer année 2023-24
df_aeqi['ATTENDANCE_MONTH'] = pd.to_datetime(df_aeqi['ATTENDANCE_MONTH'])
df_2324 = df_aeqi[
    (df_aeqi['ATTENDANCE_MONTH'] >= '2023-04-01') &
    (df_aeqi['ATTENDANCE_MONTH'] <= '2024-03-31')
].copy()

# Extraire TOTAL_TIME_MEDIAN par trust
df_a4 = df_2324[df_2324['MEASURE_NAME'] == 'TOTAL_TIME_MEDIAN'].copy()
df_a4['MEASURE_VALUE'] = pd.to_numeric(df_a4['MEASURE_VALUE'], errors='coerce')

# Moyenne annuelle par trust
df_a4 = df_a4.groupby(['ORG_CODE', 'ORG_NAME'])['MEASURE_VALUE'].mean().reset_index()
df_a4.columns = ['Trust_code', 'Trust_name_a4', 'A4_temps_attente_min']

# Garder seulement codes 3 caractères
df_a4 = df_a4[df_a4['Trust_code'].astype(str).str.len() == 3]

print(f"\n✅ A4 chargé : {len(df_a4)} trusts")
print(f"\nAperçu :")
print(df_a4.head(3).to_string())
print(f"\nStats A4 (minutes) :")
print(df_a4['A4_temps_attente_min'].describe())

Téléchargement AEQI...
✅ AEQI chargé : (96693, 7)

✅ A4 chargé : 151 trusts

Aperçu :
  Trust_code                        Trust_name_a4  A4_temps_attente_min
1        91Q        NHS KENT AND MEDWAY ICB - 91Q                169.00
5        AH6  URGENT CARE CENTRE- HURLEY GROUP HQ                114.25
6        ANH                       MALLING HEALTH                 81.75

Stats A4 (minutes) :
count    151.000000
mean     176.546973
std       55.245242
min       20.944444
25%      149.791667
50%      184.583333
75%      215.729167
max      331.500000
Name: A4_temps_attente_min, dtype: float64


In [13]:
# ============================================
# CELLULE 7 — Chargement NIDC Équipements (corrigé)
# ============================================
df_nidc = pd.read_excel(PATH_NIDC,
                        sheet_name='ICB, Imaging Network and Trust',
                        header=13)

# Garder seulement trusts avec ODS code
df_nidc = df_nidc[df_nidc['Organisation Code'].notna()].copy()
df_nidc = df_nidc[df_nidc['Organisation Code'].astype(str).str.len() == 3]

# Garder colonnes utiles
df_nidc = df_nidc[['Organisation Code', 'Organisation Name', 'Total']].copy()
df_nidc.columns = ['Trust_code', 'Trust_name_nidc', 'equipements_total']

# Convertir en numérique
df_nidc['equipements_total'] = pd.to_numeric(df_nidc['equipements_total'], errors='coerce')

# ✅ D5 ratio = équipements / lits — fusion avec df_beds
df_nidc = df_nidc.merge(df_beds[['Trust_code', 'lits_disponibles']],
                         on='Trust_code', how='left')

df_nidc['D5_ratio_equipement_lit'] = (df_nidc['equipements_total'] /
                                       df_nidc['lits_disponibles'])

print(f"✅ NIDC chargé : {len(df_nidc)} trusts")
print(f"\nAperçu :")
print(df_nidc[['Trust_code', 'equipements_total',
               'lits_disponibles', 'D5_ratio_equipement_lit']].head(5).to_string())
print(f"\nStats D5 ratio :")
print(df_nidc['D5_ratio_equipement_lit'].describe())

✅ NIDC chargé : 138 trusts

Aperçu :
  Trust_code  equipements_total  lits_disponibles  D5_ratio_equipement_lit
0        RP4               41.0        212.197802                 0.193216
1        RP6               26.0          7.032967                 3.696875
2        RAP               46.0        457.010989                 0.100654
3        RAL              155.0       1203.109890                 0.128833
4        RAN               33.0        133.736264                 0.246754

Stats D5 ratio :
count    136.000000
mean       0.135158
std        0.312679
min        0.029676
25%        0.081515
50%        0.096144
75%        0.119476
max        3.696875
Name: D5_ratio_equipement_lit, dtype: float64


In [14]:
# ============================================
# CELLULE 6 — Workforce + Ratios B7 (corrigé)
# ============================================
PATH_ORG = r'C:\Users\hp\healthsim-simulation\data\NHS Workforce Statistics, March 2024 England and Organisation.xlsx'

df_org = pd.read_excel(PATH_ORG,
                       sheet_name='3. NHSE, Org & SG - FTE',
                       header=5)

df_org = df_org[df_org['Organisation code'].notna()].copy()

df_workforce = df_org[['Organisation code', 'Organisation name',
                        'HCHS Doctors',
                        'Nurses & health visitors']].copy()

df_workforce.columns = ['Trust_code', 'Trust_name_workforce',
                         'B7_medecins_fte',
                         'B7_infirmiers_fte']

df_workforce['B7_medecins_fte']  = pd.to_numeric(df_workforce['B7_medecins_fte'], errors='coerce')
df_workforce['B7_infirmiers_fte'] = pd.to_numeric(df_workforce['B7_infirmiers_fte'], errors='coerce')

df_workforce = df_workforce.dropna(subset=['B7_medecins_fte', 'B7_infirmiers_fte'])

df_workforce = df_workforce.merge(df_beds[['Trust_code', 'lits_disponibles']],
                                   on='Trust_code', how='left')

df_workforce['ratio_medecin_lit']   = (df_workforce['B7_medecins_fte'] /
                                        df_workforce['lits_disponibles'])
df_workforce['ratio_infirmier_lit'] = (df_workforce['B7_infirmiers_fte'] /
                                        df_workforce['lits_disponibles'])
df_workforce['ratio_infirmier_medecin'] = (df_workforce['B7_infirmiers_fte'] /
                                            df_workforce['B7_medecins_fte'])

# ✅ Nettoyage valeurs infinies et NaN
import numpy as np
df_workforce = df_workforce.replace([np.inf, -np.inf], np.nan)
df_workforce = df_workforce.dropna(subset=['ratio_medecin_lit',
                                            'ratio_infirmier_lit',
                                            'ratio_infirmier_medecin'])

print(f"✅ Workforce après nettoyage : {len(df_workforce)} trusts")
print(f"\nStats ratios corrigés :")
print(df_workforce[['ratio_medecin_lit',
                     'ratio_infirmier_lit',
                     'ratio_infirmier_medecin']].describe())

✅ Workforce après nettoyage : 185 trusts

Stats ratios corrigés :
       ratio_medecin_lit  ratio_infirmier_lit  ratio_infirmier_medecin
count         185.000000           185.000000               185.000000
mean            1.254790             3.364936                 4.061383
std             3.346524             5.113050                 6.890043
min             0.227587             1.478030                 1.155283
25%             0.777922             2.101688                 2.034533
50%             0.962282             2.471369                 2.339023
75%             1.198346             3.259895                 3.584434
max            46.084730            63.453693                82.488939


In [15]:
# ============================================
# CELLULE 8 — FUSION DE TOUS LES FICHIERS
# ============================================

# Base de fusion = SHMI (119 trusts — le plus restrictif)
# On utilise SHMI comme base car il contient exactement les trusts
# qu'on veut dans notre dataset final
df_final = df_shmi[['Trust_code', 'Trust_name_shmi',
                     'C3_mortalite_pct', 'SHMI_VALUE',
                     'spells_total', 'deces_observes']].copy()
print(f"Base SHMI : {len(df_final)} trusts")

# ── Fusion 1 : NHS Beds (A8, C2, lits_disponibles) ──
df_final = df_final.merge(
    df_beds[['Trust_code', 'lits_disponibles',
             'C2_occupation_pct', 'A8_lits_vacants_pct']],
    on='Trust_code', how='left')
print(f"Après Beds : {df_final['C2_occupation_pct'].notna().sum()} trusts avec données")

# ── Fusion 2 : CDI (C4) ──
df_final = df_final.merge(
    df_cdi[['Trust_code', 'CDI_total_cases', 'C4_infections_pct']],
    on='Trust_code', how='left')
print(f"Après CDI : {df_final['C4_infections_pct'].notna().sum()} trusts avec données")

# ── Fusion 3 : Hospital Providers (A2, A9) ──
df_final = df_final.merge(
    df_prov[['Trust_code', 'A2_dms_jours',
             'A9_taux_transfert_pct',
             'total_admissions',
             'pct_admissions_urgentes']],
    on='Trust_code', how='left')
print(f"Après Providers : {df_final['A2_dms_jours'].notna().sum()} trusts avec données")

# ── Fusion 4 : AEQI (A4) ──
df_final = df_final.merge(
    df_a4[['Trust_code', 'A4_temps_attente_min']],
    on='Trust_code', how='left')
print(f"Après AEQI : {df_final['A4_temps_attente_min'].notna().sum()} trusts avec données")

# ── Fusion 5 : Workforce (B7, ratios) ──
df_final = df_final.merge(
    df_workforce[['Trust_code', 'B7_medecins_fte', 'B7_infirmiers_fte',
                  'ratio_medecin_lit', 'ratio_infirmier_lit',
                  'ratio_infirmier_medecin']],
    on='Trust_code', how='left')
print(f"Après Workforce : {df_final['B7_medecins_fte'].notna().sum()} trusts avec données")

# ── Fusion 6 : NIDC (D5) ──
df_final = df_final.merge(
    df_nidc[['Trust_code', 'equipements_total', 'D5_ratio_equipement_lit']],
    on='Trust_code', how='left')
print(f"Après NIDC : {df_final['D5_ratio_equipement_lit'].notna().sum()} trusts avec données")

print(f"\n✅ Dataset fusionné : {len(df_final)} lignes × {len(df_final.columns)} colonnes")
print(f"\nColonnes :")
for col in df_final.columns:
    print(f"  - {col}")
print(f"\nValeurs manquantes :")
print(df_final.isnull().sum())

Base SHMI : 119 trusts
Après Beds : 119 trusts avec données
Après CDI : 118 trusts avec données
Après Providers : 104 trusts avec données
Après AEQI : 119 trusts avec données
Après Workforce : 119 trusts avec données
Après NIDC : 119 trusts avec données

✅ Dataset fusionné : 119 lignes × 23 colonnes

Colonnes :
  - Trust_code
  - Trust_name_shmi
  - C3_mortalite_pct
  - SHMI_VALUE
  - spells_total
  - deces_observes
  - lits_disponibles
  - C2_occupation_pct
  - A8_lits_vacants_pct
  - CDI_total_cases
  - C4_infections_pct
  - A2_dms_jours
  - A9_taux_transfert_pct
  - total_admissions
  - pct_admissions_urgentes
  - A4_temps_attente_min
  - B7_medecins_fte
  - B7_infirmiers_fte
  - ratio_medecin_lit
  - ratio_infirmier_lit
  - ratio_infirmier_medecin
  - equipements_total
  - D5_ratio_equipement_lit

Valeurs manquantes :
Trust_code                  0
Trust_name_shmi             0
C3_mortalite_pct            1
SHMI_VALUE                  1
spells_total                1
deces_observes  

In [16]:
# ============================================
# CELLULE 9 — NETTOYAGE VALEURS MANQUANTES
# ============================================

print(f"Dataset avant nettoyage : {len(df_final)} trusts")
print(f"Valeurs manquantes par colonne :")
print(df_final.isnull().sum()[df_final.isnull().sum() > 0])

# Supprimer les lignes avec valeurs manquantes
# car on ne peut pas entraîner XGBoost avec des NaN
df_final = df_final.dropna()

print(f"\n✅ Dataset après nettoyage : {len(df_final)} trusts")
print(f"\nVérification — valeurs manquantes restantes :")
print(df_final.isnull().sum().sum())

print(f"\nAperçu du dataset final :")
print(df_final[['Trust_code', 'Trust_name_shmi',
                'C2_occupation_pct', 'C3_mortalite_pct',
                'A2_dms_jours', 'A4_temps_attente_min',
                'ratio_medecin_lit']].head(5).to_string())

print(f"\nStats générales :")
features = ['A2_dms_jours', 'A4_temps_attente_min', 
            'A8_lits_vacants_pct', 'A9_taux_transfert_pct',
            'C2_occupation_pct', 'C3_mortalite_pct',
            'C4_infections_pct', 'ratio_medecin_lit',
            'ratio_infirmier_lit', 'ratio_infirmier_medecin',
            'D5_ratio_equipement_lit', 'pct_admissions_urgentes']
print(df_final[features].describe())

Dataset avant nettoyage : 119 trusts
Valeurs manquantes par colonne :
C3_mortalite_pct            1
SHMI_VALUE                  1
spells_total                1
deces_observes              1
C4_infections_pct           1
A2_dms_jours               15
A9_taux_transfert_pct      15
total_admissions           15
pct_admissions_urgentes    15
dtype: int64

✅ Dataset après nettoyage : 103 trusts

Vérification — valeurs manquantes restantes :
0

Aperçu du dataset final :
  Trust_code                                                 Trust_name_shmi  C2_occupation_pct  C3_mortalite_pct  A2_dms_jours  A4_temps_attente_min  ratio_medecin_lit
0        RCF                                   AIREDALE NHS FOUNDATION TRUST          86.250102          3.393244      4.027666            222.375000           0.834860
1        RTK           ASHFORD AND ST PETER'S HOSPITALS NHS FOUNDATION TRUST          86.468412          2.796784      3.144720            210.750000           1.256611
2        RF4  BARKING, H

In [17]:
# ============================================
# CELLULE 10 — CALCUL DU LABEL STATUT
# ============================================
# Références officielles :
# - Seuil C2 92% : NHS Operational Planning Guidance 2023-24
# - Seuil SHMI 1.15 : NHS Digital SHMI Methodology
# - Seuil A4 240 min : NHS Constitution Standard (4 heures)
# - Seuil A2 4.5 jours : NHS HES Annual Statistics 2023-24
# - Seuil A9 5% : calculé depuis Hospital Providers 2023-24
# - Règle agrégation : CQC NHS Ratings Methodology (2024)

def calculer_statut(row):
    C2   = row['C2_occupation_pct']      # depuis df_beds
    SHMI = row['SHMI_VALUE']             # depuis df_shmi
    A2   = row['A2_dms_jours']           # depuis df_prov
    A4   = row['A4_temps_attente_min']   # depuis df_a4
    A9   = row['A9_taux_transfert_pct']  # depuis df_prov

    critique  = 0
    attention = 0

    # C2 — seuil NHS 92%
    if C2 > 92:    critique  += 1
    elif C2 > 85:  attention += 1

    # SHMI — seuil NHS 1.15
    if SHMI > 1.15:   critique  += 1
    elif SHMI > 1.0:  attention += 1

    # A2 — seuil 4.5 jours
    if A2 > 7:     critique  += 1
    elif A2 > 4.5: attention += 1

    # A4 — seuil NHS Constitution 240 min
    if A4 > 720:   critique  += 1
    elif A4 > 240: attention += 1

    # A9 — seuil 5%
    if A9 > 5:    critique  += 1
    elif A9 > 2:  attention += 1

    # Règle CQC NHS (2024)
    if critique >= 2:
        return 0  # Critique
    elif critique >= 1 or attention >= 2:
        return 1  # Attention
    else:
        return 2  # Normal

# Application sur df_final
df_final['statut'] = df_final.apply(calculer_statut, axis=1)

# Mapping lisible
statut_map = {0: 'Critique', 1: 'Attention', 2: 'Normal'}
df_final['statut_label'] = df_final['statut'].map(statut_map)

print("✅ Labels calculés !")
print(f"\nDistribution des statuts :")
print(df_final['statut_label'].value_counts())
print(f"\nEn pourcentage :")
print((df_final['statut_label'].value_counts(normalize=True) * 100).round(1))

✅ Labels calculés !

Distribution des statuts :
statut_label
Attention    73
Critique     20
Normal       10
Name: count, dtype: int64

En pourcentage :
statut_label
Attention    70.9
Critique     19.4
Normal        9.7
Name: proportion, dtype: float64


In [18]:
# ============================================
# CELLULE 11 — FONCTION CHARGEMENT PAR ANNÉE
# ============================================
import zipfile, os, requests, io
import numpy as np

def charger_annee(annee, paths):
    """
    Charge et fusionne tous les fichiers NHS pour une année donnée
    annee : str ex '2023-24' ou '2022-23'
    paths : dict avec les chemins des fichiers
    """
    print(f"\n{'='*50}")
    print(f"Chargement année {annee}...")
    print(f"{'='*50}")
    
    # ── 1. NHS Beds ──
    df_beds = pd.read_excel(paths['beds'], 
                            sheet_name='NHS Trust by Sector', header=14)
    df_beds = df_beds[['Org Code', 'Org Name', 'Total ', 'Total .1', 'Total .2']].copy()
    df_beds.columns = ['Trust_code', 'Trust_name', 'lits_disponibles', 
                       'lits_occupes', 'C2_occupation_pct']
    df_beds = df_beds[df_beds['Trust_code'].astype(str).str.len() == 3]
    df_beds['lits_disponibles']  = pd.to_numeric(df_beds['lits_disponibles'], errors='coerce')
    df_beds['C2_occupation_pct'] = pd.to_numeric(df_beds['C2_occupation_pct'], errors='coerce') * 100
    df_beds['A8_lits_vacants_pct'] = 100 - df_beds['C2_occupation_pct']
    df_beds = df_beds.dropna(subset=['lits_disponibles', 'C2_occupation_pct'])
    print(f"✅ Beds : {len(df_beds)} trusts")

    # ── 2. NHS SHMI ──
    shmi_extract = f'C:\\Users\\hp\\healthsim-simulation\\data\\SHMI_{annee}'
    with zipfile.ZipFile(paths['shmi'], 'r') as z:
        z.extractall(shmi_extract)
    
    shmi_file = None
    for root, dirs, files in os.walk(shmi_extract):
        for f in files:
            if 'trust_level' in f.lower():
                shmi_file = os.path.join(root, f)
    
    df_shmi = pd.read_csv(shmi_file)
    df_shmi = df_shmi[['PROVIDER_CODE', 'PROVIDER_NAME', 'SHMI_VALUE', 
                        'SHMI_BANDING', 'OBSERVED', 'EXPECTED', 'SPELLS']].copy()
    df_shmi.columns = ['Trust_code', 'Trust_name_shmi', 'SHMI_VALUE', 
                       'SHMI_banding', 'deces_observes', 'deces_attendus', 'spells_total']
    for col in ['SHMI_VALUE', 'deces_observes', 'deces_attendus', 'spells_total']:
        df_shmi[col] = pd.to_numeric(df_shmi[col], errors='coerce')
    df_shmi['C3_mortalite_pct'] = df_shmi['deces_observes'] / df_shmi['spells_total'] * 100
    print(f"✅ SHMI : {len(df_shmi)} trusts")

    # ── 3. CDI ──
    df_cdi_raw = pd.read_excel(paths['cdi'], engine='odf', 
                                sheet_name='data', header=2)
    df_cdi_raw.columns = ['Collection', 'Org_type', 'Org_code', 'Org_name',
                           'UKHSA', 'ICB', 'Region', 'Year', 'Month',
                           'Metric', 'Figure', 'Signed', 'Changed']
    df_cdi_filtered = df_cdi_raw[
        (df_cdi_raw['Org_type'] == 'NHS acute trust') &
        (df_cdi_raw['Metric'] == 'Total cases')
    ].copy()
    df_cdi_filtered['Figure'] = pd.to_numeric(df_cdi_filtered['Figure'], errors='coerce')
    df_cdi = df_cdi_filtered.groupby(['Org_code', 'Org_name'])['Figure'].sum().reset_index()
    df_cdi.columns = ['Trust_code', 'Trust_name_cdi', 'CDI_total_cases']
    df_cdi = df_cdi.merge(df_shmi[['Trust_code', 'spells_total']], on='Trust_code', how='left')
    df_cdi['C4_infections_pct'] = df_cdi['CDI_total_cases'] / df_cdi['spells_total'] * 100
    print(f"✅ CDI : {len(df_cdi)} trusts")

    # ── 4. Hospital Providers ──
    df_prov_raw = pd.read_excel(paths['prov'], 
                                 sheet_name='Hospital Providers', header=16)
    df_prov = df_prov_raw[['Hospital provider code and description†',
                            'Unnamed: 1', 'Finished Admission Episodes',
                            'Emergency \n(FAE)', 'Mean length of stay \n(Days)',
                            'FCE bed days', 'Emergency \n(FAE).1',
                            'Elective\n(FAE)', 'Other\n(FAE)']].copy()
    df_prov.columns = ['Trust_code', 'Trust_name_prov', 'total_admissions',
                       'emergency_fae_admission', 'A2_dms_jours', 'fce_bed_days',
                       'emergency_fae_discharge', 'elective_fae_discharge', 'other_fae_discharge']
    df_prov = df_prov[df_prov['Trust_code'].astype(str).str.len() == 3]
    for col in ['total_admissions', 'A2_dms_jours', 'other_fae_discharge', 
                'emergency_fae_admission']:
        df_prov[col] = pd.to_numeric(df_prov[col], errors='coerce')
    df_prov['A9_taux_transfert_pct'] = df_prov['other_fae_discharge'] / df_prov['total_admissions'] * 100
    df_prov['pct_admissions_urgentes'] = df_prov['emergency_fae_admission'] / df_prov['total_admissions'] * 100
    df_prov = df_prov.dropna(subset=['A2_dms_jours', 'A9_taux_transfert_pct'])
    print(f"✅ Hospital Providers : {len(df_prov)} trusts")

    # ── 5. AEQI ──
    url_aeqi = paths['aeqi']
    response = requests.get(url_aeqi, timeout=120)
    df_aeqi = pd.read_csv(io.StringIO(response.content.decode('utf-8')))
    df_aeqi['ATTENDANCE_MONTH'] = pd.to_datetime(df_aeqi['ATTENDANCE_MONTH'])
    df_a4 = df_aeqi[df_aeqi['MEASURE_NAME'] == 'TOTAL_TIME_MEDIAN'].copy()
    df_a4['MEASURE_VALUE'] = pd.to_numeric(df_a4['MEASURE_VALUE'], errors='coerce')
    df_a4 = df_a4.groupby(['ORG_CODE', 'ORG_NAME'])['MEASURE_VALUE'].mean().reset_index()
    df_a4.columns = ['Trust_code', 'Trust_name_a4', 'A4_temps_attente_min']
    df_a4 = df_a4[df_a4['Trust_code'].astype(str).str.len() == 3]
    print(f"✅ AEQI : {len(df_a4)} trusts")

    # ── 6. Workforce ──
    df_org = pd.read_excel(paths['workforce'], 
                            sheet_name='3. NHSE, Org & SG - FTE', header=5)
    df_org = df_org[df_org['Organisation code'].notna()].copy()
    df_workforce = df_org[['Organisation code', 'Organisation name',
                            'HCHS Doctors', 'Nurses & health visitors']].copy()
    df_workforce.columns = ['Trust_code', 'Trust_name_workforce',
                             'B7_medecins_fte', 'B7_infirmiers_fte']
    df_workforce['B7_medecins_fte']  = pd.to_numeric(df_workforce['B7_medecins_fte'], errors='coerce')
    df_workforce['B7_infirmiers_fte'] = pd.to_numeric(df_workforce['B7_infirmiers_fte'], errors='coerce')
    df_workforce = df_workforce.dropna()
    df_workforce = df_workforce.merge(df_beds[['Trust_code', 'lits_disponibles']], 
                                       on='Trust_code', how='left')
    df_workforce['ratio_medecin_lit']       = df_workforce['B7_medecins_fte'] / df_workforce['lits_disponibles']
    df_workforce['ratio_infirmier_lit']     = df_workforce['B7_infirmiers_fte'] / df_workforce['lits_disponibles']
    df_workforce['ratio_infirmier_medecin'] = df_workforce['B7_infirmiers_fte'] / df_workforce['B7_medecins_fte']
    df_workforce = df_workforce.replace([np.inf, -np.inf], np.nan).dropna(
        subset=['ratio_medecin_lit', 'ratio_infirmier_lit', 'ratio_infirmier_medecin'])
    print(f"✅ Workforce : {len(df_workforce)} trusts")

    # ── 7. NIDC ──
    df_nidc = pd.read_excel(paths['nidc'], 
                             sheet_name='ICB, Imaging Network and Trust', header=13)
    df_nidc = df_nidc[df_nidc['Organisation Code'].notna()].copy()
    df_nidc = df_nidc[df_nidc['Organisation Code'].astype(str).str.len() == 3]
    df_nidc = df_nidc[['Organisation Code', 'Organisation Name', 'Total']].copy()
    df_nidc.columns = ['Trust_code', 'Trust_name_nidc', 'equipements_total']
    df_nidc['equipements_total'] = pd.to_numeric(df_nidc['equipements_total'], errors='coerce')
    df_nidc = df_nidc.merge(df_beds[['Trust_code', 'lits_disponibles']], 
                             on='Trust_code', how='left')
    df_nidc['D5_ratio_equipement_lit'] = df_nidc['equipements_total'] / df_nidc['lits_disponibles']
    print(f"✅ NIDC : {len(df_nidc)} trusts")

    # ── FUSION ──
    df = df_shmi[['Trust_code', 'Trust_name_shmi', 'C3_mortalite_pct', 
                   'SHMI_VALUE', 'spells_total', 'deces_observes']].copy()
    df = df.merge(df_beds[['Trust_code', 'lits_disponibles', 'C2_occupation_pct', 
                            'A8_lits_vacants_pct']], on='Trust_code', how='left')
    df = df.merge(df_cdi[['Trust_code', 'CDI_total_cases', 'C4_infections_pct']], 
                  on='Trust_code', how='left')
    df = df.merge(df_prov[['Trust_code', 'A2_dms_jours', 'A9_taux_transfert_pct',
                            'total_admissions', 'pct_admissions_urgentes']], 
                  on='Trust_code', how='left')
    df = df.merge(df_a4[['Trust_code', 'A4_temps_attente_min']], 
                  on='Trust_code', how='left')
    df = df.merge(df_workforce[['Trust_code', 'B7_medecins_fte', 'B7_infirmiers_fte',
                                 'ratio_medecin_lit', 'ratio_infirmier_lit', 
                                 'ratio_infirmier_medecin']], 
                  on='Trust_code', how='left')
    df = df.merge(df_nidc[['Trust_code', 'equipements_total', 'D5_ratio_equipement_lit']], 
                  on='Trust_code', how='left')
    
    # Nettoyage
    df = df.dropna()
    df['annee'] = annee
    
    print(f"\n✅ Dataset {annee} : {len(df)} trusts × {len(df.columns)} colonnes")
    return df

print("✅ Fonction définie !")

✅ Fonction définie !


In [19]:
# ============================================
# CELLULE 11 — FONCTION CHARGEMENT PAR ANNÉE
# ============================================
import zipfile, os, requests, io
import numpy as np

def charger_annee(annee, paths):
    print(f"\n{'='*50}")
    print(f"Chargement année {annee}...")
    print(f"{'='*50}")
    
    # ── 1. NHS Beds ──
    df_beds = pd.read_excel(paths['beds'], 
                            sheet_name='NHS Trust by Sector', header=14)
    df_beds = df_beds[['Org Code', 'Org Name', 'Total ', 'Total .1', 'Total .2']].copy()
    df_beds.columns = ['Trust_code', 'Trust_name', 'lits_disponibles', 
                       'lits_occupes', 'C2_occupation_pct']
    df_beds = df_beds[df_beds['Trust_code'].astype(str).str.len() == 3]
    df_beds['lits_disponibles']  = pd.to_numeric(df_beds['lits_disponibles'], errors='coerce')
    df_beds['C2_occupation_pct'] = pd.to_numeric(df_beds['C2_occupation_pct'], errors='coerce') * 100
    df_beds['A8_lits_vacants_pct'] = 100 - df_beds['C2_occupation_pct']
    df_beds = df_beds.dropna(subset=['lits_disponibles', 'C2_occupation_pct'])
    print(f"✅ Beds : {len(df_beds)} trusts")

    # ── 2. NHS SHMI ──
    shmi_extract = rf'C:\Users\hp\healthsim-simulation\data\SHMI_{annee}'
    with zipfile.ZipFile(paths['shmi'], 'r') as z:
        z.extractall(shmi_extract)
    
    shmi_file = None
    for root, dirs, files in os.walk(shmi_extract):
        for f in files:
            if ('shmi data at trust level' in f.lower() or
                'shmi_data_at_trust' in f.lower()) and f.endswith('.csv'):
                shmi_file = os.path.join(root, f)
                print(f"  Fichier SHMI trouvé : {f}")

    df_shmi = pd.read_csv(shmi_file, encoding='latin-1')
    df_shmi = df_shmi[['PROVIDER_CODE', 'PROVIDER_NAME', 'SHMI_VALUE', 
                        'SHMI_BANDING', 'OBSERVED', 'EXPECTED', 'SPELLS']].copy()
    df_shmi.columns = ['Trust_code', 'Trust_name_shmi', 'SHMI_VALUE', 
                       'SHMI_banding', 'deces_observes', 'deces_attendus', 'spells_total']
    for col in ['SHMI_VALUE', 'deces_observes', 'deces_attendus', 'spells_total']:
        df_shmi[col] = pd.to_numeric(df_shmi[col], errors='coerce')
    df_shmi['C3_mortalite_pct'] = df_shmi['deces_observes'] / df_shmi['spells_total'] * 100
    print(f"✅ SHMI : {len(df_shmi)} trusts")

    # ── 3. CDI ──
    df_cdi_raw = pd.read_excel(paths['cdi'], engine='odf', 
                                sheet_name='data', header=2)
    df_cdi_raw.columns = ['Collection', 'Org_type', 'Org_code', 'Org_name',
                           'UKHSA', 'ICB', 'Region', 'Year', 'Month',
                           'Metric', 'Figure', 'Signed', 'Changed']
    df_cdi_filtered = df_cdi_raw[
        (df_cdi_raw['Org_type'] == 'NHS acute trust') &
        (df_cdi_raw['Metric'] == 'Total cases')
    ].copy()
    df_cdi_filtered['Figure'] = pd.to_numeric(df_cdi_filtered['Figure'], errors='coerce')
    df_cdi = df_cdi_filtered.groupby(['Org_code', 'Org_name'])['Figure'].sum().reset_index()
    df_cdi.columns = ['Trust_code', 'Trust_name_cdi', 'CDI_total_cases']
    df_cdi = df_cdi.merge(df_shmi[['Trust_code', 'spells_total']], on='Trust_code', how='left')
    df_cdi['C4_infections_pct'] = df_cdi['CDI_total_cases'] / df_cdi['spells_total'] * 100
    print(f"✅ CDI : {len(df_cdi)} trusts")

    # ── 4. Hospital Providers ──
    df_prov_raw = pd.read_excel(paths['prov'],
                                 sheet_name='Hospital Providers', header=16)
    cols = df_prov_raw.columns.tolist()
    
    if 'Finished Admission Episodes' in cols:
        # Format 2023-24
        df_prov = df_prov_raw[['Hospital provider code and description†',
                                'Unnamed: 1', 'Finished Admission Episodes',
                                'Emergency \n(FAE)', 'Mean length of stay \n(Days)',
                                'FCE bed days', 'Emergency \n(FAE).1',
                                'Elective\n(FAE)', 'Other\n(FAE)']].copy()
    else:
        # Format 2022-23
        df_prov = df_prov_raw[['Hospital provider code and description†',
                                'Unnamed: 1', 'Admissions',
                                'Emergency', 'Mean length of stay',
                                'FCE bed days', 'Emergency.1',
                                'Elective', 'Other']].copy()

    df_prov.columns = ['Trust_code', 'Trust_name_prov', 'total_admissions',
                       'emergency_fae_admission', 'A2_dms_jours', 'fce_bed_days',
                       'emergency_fae_discharge', 'elective_fae_discharge', 'other_fae_discharge']
    df_prov = df_prov[df_prov['Trust_code'].astype(str).str.len() == 3]
    for col in ['total_admissions', 'A2_dms_jours', 'other_fae_discharge',
                'emergency_fae_admission']:
        df_prov[col] = pd.to_numeric(df_prov[col], errors='coerce')
    df_prov['A9_taux_transfert_pct'] = df_prov['other_fae_discharge'] / df_prov['total_admissions'] * 100
    df_prov['pct_admissions_urgentes'] = df_prov['emergency_fae_admission'] / df_prov['total_admissions'] * 100
    df_prov = df_prov.dropna(subset=['A2_dms_jours', 'A9_taux_transfert_pct'])
    print(f"✅ Hospital Providers : {len(df_prov)} trusts")

    # ── 5. AEQI ──
    aeqi_path = paths['aeqi']
    if aeqi_path.startswith('http'):
        response = requests.get(aeqi_path, timeout=120)
        df_aeqi = pd.read_csv(io.StringIO(response.content.decode('utf-8')))
    else:
        df_aeqi = pd.read_csv(aeqi_path)

    df_aeqi['ATTENDANCE_MONTH'] = pd.to_datetime(df_aeqi['ATTENDANCE_MONTH'])
    df_a4 = df_aeqi[df_aeqi['MEASURE_NAME'] == 'TOTAL_TIME_MEDIAN'].copy()
    df_a4['MEASURE_VALUE'] = pd.to_numeric(df_a4['MEASURE_VALUE'], errors='coerce')
    df_a4 = df_a4.groupby(['ORG_CODE', 'ORG_NAME'])['MEASURE_VALUE'].mean().reset_index()
    df_a4.columns = ['Trust_code', 'Trust_name_a4', 'A4_temps_attente_min']
    df_a4 = df_a4[df_a4['Trust_code'].astype(str).str.len() == 3]
    print(f"✅ AEQI : {len(df_a4)} trusts")

    # ── 6. Workforce ──
    df_org = pd.read_excel(paths['workforce'], 
                            sheet_name='3. NHSE, Org & SG - FTE', header=5)
    df_org = df_org[df_org['Organisation code'].notna()].copy()
    df_workforce = df_org[['Organisation code', 'Organisation name',
                            'HCHS Doctors', 'Nurses & health visitors']].copy()
    df_workforce.columns = ['Trust_code', 'Trust_name_workforce',
                             'B7_medecins_fte', 'B7_infirmiers_fte']
    df_workforce['B7_medecins_fte']  = pd.to_numeric(df_workforce['B7_medecins_fte'], errors='coerce')
    df_workforce['B7_infirmiers_fte'] = pd.to_numeric(df_workforce['B7_infirmiers_fte'], errors='coerce')
    df_workforce = df_workforce.dropna()
    df_workforce = df_workforce.merge(df_beds[['Trust_code', 'lits_disponibles']], 
                                       on='Trust_code', how='left')
    df_workforce['ratio_medecin_lit']       = df_workforce['B7_medecins_fte'] / df_workforce['lits_disponibles']
    df_workforce['ratio_infirmier_lit']     = df_workforce['B7_infirmiers_fte'] / df_workforce['lits_disponibles']
    df_workforce['ratio_infirmier_medecin'] = df_workforce['B7_infirmiers_fte'] / df_workforce['B7_medecins_fte']
    df_workforce = df_workforce.replace([np.inf, -np.inf], np.nan).dropna(
        subset=['ratio_medecin_lit', 'ratio_infirmier_lit', 'ratio_infirmier_medecin'])
    print(f"✅ Workforce : {len(df_workforce)} trusts")

    # ── 7. NIDC ──
    df_nidc = pd.read_excel(paths['nidc'], 
                             sheet_name='ICB, Imaging Network and Trust', header=13)
    df_nidc = df_nidc[df_nidc['Organisation Code'].notna()].copy()
    df_nidc = df_nidc[df_nidc['Organisation Code'].astype(str).str.len() == 3]
    df_nidc = df_nidc[['Organisation Code', 'Organisation Name', 'Total']].copy()
    df_nidc.columns = ['Trust_code', 'Trust_name_nidc', 'equipements_total']
    df_nidc['equipements_total'] = pd.to_numeric(df_nidc['equipements_total'], errors='coerce')
    df_nidc = df_nidc.merge(df_beds[['Trust_code', 'lits_disponibles']], 
                             on='Trust_code', how='left')
    df_nidc['D5_ratio_equipement_lit'] = df_nidc['equipements_total'] / df_nidc['lits_disponibles']
    print(f"✅ NIDC : {len(df_nidc)} trusts")

    # ── FUSION ──
    df = df_shmi[['Trust_code', 'Trust_name_shmi', 'C3_mortalite_pct', 
                   'SHMI_VALUE', 'spells_total', 'deces_observes']].copy()
    df = df.merge(df_beds[['Trust_code', 'lits_disponibles', 'C2_occupation_pct', 
                            'A8_lits_vacants_pct']], on='Trust_code', how='left')
    df = df.merge(df_cdi[['Trust_code', 'CDI_total_cases', 'C4_infections_pct']], 
                  on='Trust_code', how='left')
    df = df.merge(df_prov[['Trust_code', 'A2_dms_jours', 'A9_taux_transfert_pct',
                            'total_admissions', 'pct_admissions_urgentes']], 
                  on='Trust_code', how='left')
    df = df.merge(df_a4[['Trust_code', 'A4_temps_attente_min']], 
                  on='Trust_code', how='left')
    df = df.merge(df_workforce[['Trust_code', 'B7_medecins_fte', 'B7_infirmiers_fte',
                                 'ratio_medecin_lit', 'ratio_infirmier_lit', 
                                 'ratio_infirmier_medecin']], 
                  on='Trust_code', how='left')
    df = df.merge(df_nidc[['Trust_code', 'equipements_total', 'D5_ratio_equipement_lit']], 
                  on='Trust_code', how='left')
    
    df = df.dropna()
    df['annee'] = annee
    
    print(f"\n✅ Dataset {annee} : {len(df)} trusts × {len(df.columns)} colonnes")
    return df

print("✅ Fonction définie !")

✅ Fonction définie !


In [20]:
# ============================================
# CELLULE 12 — CHARGEMENT 2022-23 ET 2023-24
# ============================================
DATA = r'C:\Users\hp\healthsim-simulation\data'

# ── Chemins 2023-24 ──
paths_2324 = {
    'beds'      : rf'{DATA}\Beds-Open-Overnight-Web_File-Q4-2023-24-Final.xlsx',
    'shmi'      : rf'{DATA}\SHMI data, Apr23-Mar24.zip',
    'cdi'       : rf'{DATA}\Table_6_CDI_april_2023_to_march_2024-os.ods',
    'prov'      : rf'{DATA}\hosp-epis-stat-admi-hosp-prov-2023-24-tab.xlsx',
    'aeqi'      : 'https://files.digital.nhs.uk/23/C1A9E0/aeqi_open_data_2024_03.csv',
    'workforce' : rf'{DATA}\NHS Workforce Statistics, March 2024 England and Organisation.xlsx',
    'nidc'      : rf'{DATA}\National-Imaging-Data-Collection-Asset-Count-2023-24-v1.xlsx',
}

# ── Chemins 2022-23 ──
paths_2223 = {
    'beds'      : rf'{DATA}\Beds-Open-Overnight-Web_File-Q4-2022-23-Final.xlsx',
    'shmi'      : rf'{DATA}\SHMI data files, Apr22-Mar23.zip',
    'cdi'       : rf'{DATA}\Table-6-CDI-march-2023.ods',
    'prov'      : rf'{DATA}\hosp-epis-stat-admi-hosp-prov-2022-23-tab_v2.xlsx',
    'aeqi'      : rf'{DATA}\aeqi_open_data_2023_03.csv',
    'workforce' : rf'{DATA}\NHS Workforce Statistics, March 2023 England and Organisation.xlsx',
    'nidc'      : rf'{DATA}\National-Imaging-Data-Collection-Asset-Count-2022-23-v1.xlsx',
}

# ── Chargement ──
df_2324 = charger_annee('2023-24', paths_2324)
df_2223 = charger_annee('2022-23', paths_2223)

# ── Combinaison ──
df_combined = pd.concat([df_2324, df_2223], ignore_index=True)

print(f"\n{'='*50}")
print(f"✅ Dataset combiné : {len(df_combined)} trusts × {len(df_combined.columns)} colonnes")
print(f"\nDistribution par année :")
print(df_combined['annee'].value_counts())


Chargement année 2023-24...
✅ Beds : 185 trusts
  Fichier SHMI trouvé : SHMI_data_at_trust_level_Apr23-Mar24_csv.csv
✅ SHMI : 119 trusts
✅ CDI : 136 trusts
✅ Hospital Providers : 150 trusts
✅ AEQI : 154 trusts
✅ Workforce : 185 trusts
✅ NIDC : 138 trusts

✅ Dataset 2023-24 : 103 trusts × 24 colonnes

Chargement année 2022-23...
✅ Beds : 187 trusts
  Fichier SHMI trouvé : SHMI data at trust level, Apr22-Mar23 (csv).csv
✅ SHMI : 120 trusts
✅ CDI : 135 trusts
✅ Hospital Providers : 144 trusts
✅ AEQI : 154 trusts
✅ Workforce : 187 trusts
✅ NIDC : 137 trusts

✅ Dataset 2022-23 : 103 trusts × 24 colonnes

✅ Dataset combiné : 206 trusts × 24 colonnes

Distribution par année :
annee
2023-24    103
2022-23    103
Name: count, dtype: int64


In [21]:
# CELLULE DIAGNOSTIC — Voir le contenu du ZIP SHMI 2022-23
import zipfile

with zipfile.ZipFile(r'C:\Users\hp\healthsim-simulation\data\SHMI data files, Apr22-Mar23.zip', 'r') as z:
    print("Fichiers dans le ZIP 2022-23 :")
    for f in z.namelist():
        if '.csv' in f.lower():
            print(f"  - {f}")

Fichiers dans le ZIP 2022-23 :
  - SHMI contextual indicators/Admission method/Deaths within 30 days for elective admissions, Apr22-Mar23 (csv).csv
  - SHMI contextual indicators/Admission method/Deaths within 30 days for non-elective admissions, Apr22-Mar23 (csv).csv
  - SHMI contextual indicators/COVID-19 activity/Percentage of provider spells with COVID-19 coding, Apr22-Mar23 (csv).csv
  - SHMI contextual indicators/COVID-19 activity/Provider spells compared to the pre-pandemic period, Apr22-Mar23 (csv).csv
  - SHMI contextual indicators/Deprivation/Deaths split by deprivation quintile, Apr22-Mar23 (csv).csv
  - SHMI contextual indicators/Deprivation/Provider spells split by deprivation quintile, Apr22-Mar23 (csv).csv
  - SHMI contextual indicators/Depth of coding/Mean depth of coding for elective admissions, Apr22-Mar23 (csv).csv
  - SHMI contextual indicators/Depth of coding/Mean depth of coding for non-elective admissions, Apr22-Mar23 (csv).csv
  - SHMI contextual indicators/In a

In [22]:
# CELLULE DIAGNOSTIC — Colonnes Hospital Providers 2022-23
df_prov_raw_2223 = pd.read_excel(
    r'C:\Users\hp\healthsim-simulation\data\hosp-epis-stat-admi-hosp-prov-2022-23-tab_v2.xlsx',
    sheet_name='Hospital Providers', header=16)

print("Colonnes disponibles :")
for col in df_prov_raw_2223.columns:
    print(f"  - '{col}'")

Colonnes disponibles :
  - 'Hospital provider code and description†'
  - 'Unnamed: 1'
  - 'Unnamed: 2'
  - 'Unnamed: 3'
  - 'Unnamed: 4'
  - 'Unnamed: 5'
  - 'Unnamed: 6'
  - 'Finished consultant episodes'
  - 'Admissions'
  - 'Male'
  - 'Female'
  - 'Gender Unknown'
  - 'Emergency'
  - 'Waiting list'
  - 'Planned'
  - 'Other Admission Method'
  - 'Mean time waited'
  - 'Median time waited'
  - 'Mean length of stay'
  - 'Median length of stay'
  - 'Mean age'
  - 'Age 0'
  - 'Age 1-4'
  - 'Age 5-9'
  - 'Age 10-14'
  - 'Age 15'
  - 'Age 16'
  - 'Age 17'
  - 'Age 18'
  - 'Age 19'
  - 'Age 20-24'
  - 'Age 25-29'
  - 'Age 30-34'
  - 'Age 35-39'
  - 'Age 40-44'
  - 'Age 45-49'
  - 'Age 50-54'
  - 'Age 55-59'
  - 'Age 60-64'
  - 'Age 65-69'
  - 'Age 70-74'
  - 'Age 75-79'
  - 'Age 80-84'
  - 'Age 85-89'
  - 'Age 90+'
  - 'Day case'
  - 'FCE bed days'
  - 'Emergency.1'
  - 'Elective'
  - 'Other'


In [23]:
# ============================================
# CELLULE 13 — CALCUL DU LABEL SUR df_combined
# ============================================
def calculer_statut(row):
    C2   = row['C2_occupation_pct']
    SHMI = row['SHMI_VALUE']
    A2   = row['A2_dms_jours']
    A4   = row['A4_temps_attente_min']
    A9   = row['A9_taux_transfert_pct']

    critique  = 0
    attention = 0

    if C2 > 92:    critique  += 1
    elif C2 > 85:  attention += 1

    if SHMI > 1.15:   critique  += 1
    elif SHMI > 1.0:  attention += 1

    if A2 > 7:     critique  += 1
    elif A2 > 4.5: attention += 1

    if A4 > 720:   critique  += 1
    elif A4 > 240: attention += 1

    if A9 > 5:    critique  += 1
    elif A9 > 2:  attention += 1

    if critique >= 2:
        return 0  # Critique
    elif critique >= 1 or attention >= 2:
        return 1  # Attention
    else:
        return 2  # Normal

df_combined['statut'] = df_combined.apply(calculer_statut, axis=1)
statut_map = {0: 'Critique', 1: 'Attention', 2: 'Normal'}
df_combined['statut_label'] = df_combined['statut'].map(statut_map)

print("✅ Labels calculés !")
print(f"\nDistribution globale :")
print(df_combined['statut_label'].value_counts())
print(f"\nEn % :")
print((df_combined['statut_label'].value_counts(normalize=True) * 100).round(1))
print(f"\nDistribution par année :")
print(df_combined.groupby(['annee', 'statut_label']).size().unstack(fill_value=0))

✅ Labels calculés !

Distribution globale :
statut_label
Attention    160
Critique      30
Normal        16
Name: count, dtype: int64

En % :
statut_label
Attention    77.7
Critique     14.6
Normal        7.8
Name: proportion, dtype: float64

Distribution par année :
statut_label  Attention  Critique  Normal
annee                                    
2022-23              86        10       7
2023-24              74        20       9


In [24]:
# ============================================
# CELLULE 14 — SAUVEGARDE DU DATASET FINAL
# ============================================
import os

# Créer le dossier datasets
os.makedirs(r'C:\Users\hp\healthsim-simulation\data\datasets', exist_ok=True)

# Features finales — toutes calculables dans HealthSim
FEATURES = [
    'A2_dms_jours',           # calculé depuis simulation
    'A4_temps_attente_min',   # calculé par Erlang-C
    'A8_lits_vacants_pct',    # calculé depuis lits
    'A9_taux_transfert_pct',  # saisi par chef service
    'C2_occupation_pct',      # calculé depuis lits
    'C3_mortalite_pct',       # saisi par chef service
    'C4_infections_pct',      # saisi par chef service
    'ratio_medecin_lit',      # médecins / lits
    'ratio_infirmier_lit',    # infirmiers / lits
    'ratio_infirmier_medecin',# infirmiers / médecins
    'D5_ratio_equipement_lit' # équipements / lits
]

# Sauvegarder le dataset
path_dataset = r'C:\Users\hp\healthsim-simulation\data\datasets\nhs_dataset_final.csv'
df_combined.to_csv(path_dataset, index=False)

print(f"✅ Dataset sauvegardé !")
print(f"Chemin : data\\datasets\\nhs_dataset_final.csv")
print(f"Dimensions : {df_combined.shape}")
print(f"\nFeatures pour le modèle : {len(FEATURES)}")
for f in FEATURES:
    print(f"  - {f}")
print(f"\nDistribution labels :")
print(df_combined['statut_label'].value_counts())

✅ Dataset sauvegardé !
Chemin : data\datasets\nhs_dataset_final.csv
Dimensions : (206, 26)

Features pour le modèle : 11
  - A2_dms_jours
  - A4_temps_attente_min
  - A8_lits_vacants_pct
  - A9_taux_transfert_pct
  - C2_occupation_pct
  - C3_mortalite_pct
  - C4_infections_pct
  - ratio_medecin_lit
  - ratio_infirmier_lit
  - ratio_infirmier_medecin
  - D5_ratio_equipement_lit

Distribution labels :
statut_label
Attention    160
Critique      30
Normal        16
Name: count, dtype: int64


In [25]:
# Voir le dataset complet
df_combined[['Trust_code', 'Trust_name_shmi', 'annee'] + FEATURES + ['statut_label']].head(20) 

,Trust_code,Trust_name_shmi,annee,A2_dms_jours,A4_temps_attente_min,A8_lits_vacants_pct,A9_taux_transfert_pct,C2_occupation_pct,C3_mortalite_pct,C4_infections_pct,ratio_medecin_lit,ratio_infirmier_lit,ratio_infirmier_medecin,D5_ratio_equipement_lit,statut_label
0,RCF,AIREDALE NHS FOUNDATION TRUST,2023-24,4.027666,220.58,13.749898,2.049486,86.250102,3.393244,0.368764,0.834860,1.778592,2.130406,0.078974,Attention
1,RTK,ASHFORD AND ST PETER'S HOSPITALS NHS FOUNDATIO...,2023-24,3.144720,222.16,13.531588,6.905407,86.468412,2.796784,0.121451,1.256611,2.226957,1.772193,0.117908,Attention
2,RF4,"BARKING, HAVERING AND REDBRIDGE UNIVERSITY HOS...",2023-24,3.516111,309.10,5.234281,0.793557,94.765719,2.283519,0.091513,1.216217,2.135719,1.756035,0.084675,Attention
3,RFF,BARNSLEY HOSPITAL NHS FOUNDATION TRUST,2023-24,3.644835,211.12,7.592712,2.914826,92.407288,3.227711,0.153513,0.828729,2.024446,2.442832,0.081929,Attention
4,R1H,BARTS HEALTH NHS TRUST,2023-24,5.373930,231.44,5.070612,1.356528,94.929388,2.722792,0.139121,1.428163,2.430055,1.701524,0.113366,Attention
5,RC9,BEDFORDSHIRE HOSPITALS NHS FOUNDATION TRUST,2023-24,3.398034,180.34,6.502244,6.559870,93.497756,2.443274,0.094003,1.051985,1.802187,1.713130,0.084124,Critique
6,RXL,BLACKPOOL TEACHING HOSPITALS NHS FOUNDATION TRUST,2023-24,5.374062,328.64,4.060418,1.468585,95.939582,3.868265,0.203679,0.821373,2.944157,3.584434,0.084942,Attention
7,RMC,BOLTON NHS FOUNDATION TRUST,2023-24,4.061010,210.88,10.216234,7.915453,89.783766,2.998066,0.335854,0.849720,2.469066,2.905739,0.087678,Attention
8,RAE,BRADFORD TEACHING HOSPITALS NHS FOUNDATION TRUST,2023-24,3.094164,215.08,7.870717,12.946923,92.129283,2.134590,0.055837,1.127823,2.119832,1.879579,0.119065,Critique
9,RXQ,BUCKINGHAMSHIRE HEALTHCARE NHS TRUST,2023-24,3.364101,301.84,4.403357,7.593318,95.596643,2.194592,0.105993,1.391809,3.338076,2.398372,0.151675,Critique


In [26]:
# Voir TOUT le dataset
pd.set_option('display.max_rows', 206)
df_combined[['Trust_code', 'Trust_name_shmi', 'annee'] + FEATURES + ['statut_label']]

,Trust_code,Trust_name_shmi,annee,A2_dms_jours,A4_temps_attente_min,A8_lits_vacants_pct,A9_taux_transfert_pct,C2_occupation_pct,C3_mortalite_pct,C4_infections_pct,ratio_medecin_lit,ratio_infirmier_lit,ratio_infirmier_medecin,D5_ratio_equipement_lit,statut_label
0,RCF,AIREDALE NHS FOUNDATION TRUST,2023-24,4.027666,220.580000,13.749898,2.049486,86.250102,3.393244,0.368764,0.834860,1.778592,2.130406,0.078974,Attention
1,RTK,ASHFORD AND ST PETER'S HOSPITALS NHS FOUNDATIO...,2023-24,3.144720,222.160000,13.531588,6.905407,86.468412,2.796784,0.121451,1.256611,2.226957,1.772193,0.117908,Attention
2,RF4,"BARKING, HAVERING AND REDBRIDGE UNIVERSITY HOS...",2023-24,3.516111,309.100000,5.234281,0.793557,94.765719,2.283519,0.091513,1.216217,2.135719,1.756035,0.084675,Attention
3,RFF,BARNSLEY HOSPITAL NHS FOUNDATION TRUST,2023-24,3.644835,211.120000,7.592712,2.914826,92.407288,3.227711,0.153513,0.828729,2.024446,2.442832,0.081929,Attention
4,R1H,BARTS HEALTH NHS TRUST,2023-24,5.373930,231.440000,5.070612,1.356528,94.929388,2.722792,0.139121,1.428163,2.430055,1.701524,0.113366,Attention
5,RC9,BEDFORDSHIRE HOSPITALS NHS FOUNDATION TRUST,2023-24,3.398034,180.340000,6.502244,6.559870,93.497756,2.443274,0.094003,1.051985,1.802187,1.713130,0.084124,Critique
6,RXL,BLACKPOOL TEACHING HOSPITALS NHS FOUNDATION TRUST,2023-24,5.374062,328.640000,4.060418,1.468585,95.939582,3.868265,0.203679,0.821373,2.944157,3.584434,0.084942,Attention
7,RMC,BOLTON NHS FOUNDATION TRUST,2023-24,4.061010,210.880000,10.216234,7.915453,89.783766,2.998066,0.335854,0.849720,2.469066,2.905739,0.087678,Attention
8,RAE,BRADFORD TEACHING HOSPITALS NHS FOUNDATION TRUST,2023-24,3.094164,215.080000,7.870717,12.946923,92.129283,2.134590,0.055837,1.127823,2.119832,1.879579,0.119065,Critique
9,RXQ,BUCKINGHAMSHIRE HEALTHCARE NHS TRUST,2023-24,3.364101,301.840000,4.403357,7.593318,95.596643,2.194592,0.105993,1.391809,3.338076,2.398372,0.151675,Critique


In [27]:
# ============================================
# CELLULE — NETTOYAGE ET RENOMMAGE
# ============================================

# Supprimer spells_total et Pct_Admissions_Urgentes
df_combined = df_combined.drop(columns=['spells_total', 'pct_admissions_urgentes'])

# Renommer toutes les colonnes
df_combined = df_combined.rename(columns={
    'Trust_code'              : 'Code_Hopital',
    'Trust_name_shmi'         : 'Nom_Hopital',
    'C3_mortalite_pct'        : 'Taux_Mortalite',
    'SHMI_VALUE'              : 'SHMI',
    'deces_observes'          : 'Nombre_Deces',
    'lits_disponibles'        : 'Lits_Disponibles',
    'C2_occupation_pct'       : 'Taux_Occupation',
    'A8_lits_vacants_pct'     : 'Taux_Lits_Vacants',
    'CDI_total_cases'         : 'Total_Infections_CDI',
    'C4_infections_pct'       : 'Taux_Infections',
    'A2_dms_jours'            : 'Duree_Moyenne_Sejour_Jours',
    'A9_taux_transfert_pct'   : 'Taux_Transfert',
    'total_admissions'        : 'Total_Admissions',
    'A4_temps_attente_min'    : 'Temps_Attente_Min',
    'B7_medecins_fte'         : 'Nombre_Medecins',
    'B7_infirmiers_fte'       : 'Nombre_Infirmiers',
    'ratio_medecin_lit'       : 'Ratio_Medecin_Lit',
    'ratio_infirmier_lit'     : 'Ratio_Infirmier_Lit',
    'ratio_infirmier_medecin' : 'Ratio_Infirmier_Medecin',
    'equipements_total'       : 'Total_Equipements',
    'D5_ratio_equipement_lit' : 'Ratio_Equipement_Lit',
    'annee'                   : 'Annee',
    'statut'                  : 'Statut',
    'statut_label'            : 'Statut_Label'
})

# Recalculer C3 avec Total_Admissions
df_combined['Taux_Mortalite'] = (
    df_combined['Nombre_Deces'] /
    df_combined['Total_Admissions'] * 100
)

# Recalculer C4 avec Total_Admissions
df_combined['Taux_Infections'] = (
    df_combined['Total_Infections_CDI'] /
    df_combined['Total_Admissions'] * 100
)

# Mettre à jour FEATURES
FEATURES = [
    'Duree_Moyenne_Sejour_Jours',
    'Temps_Attente_Min',
    'Taux_Lits_Vacants',
    'Taux_Transfert',
    'Taux_Occupation',
    'Taux_Mortalite',
    'Taux_Infections',
    'Ratio_Medecin_Lit',
    'Ratio_Infirmier_Lit',
    'Ratio_Infirmier_Medecin',
    'Ratio_Equipement_Lit'
]

# Sauvegarder
path_dataset = r'C:\Users\hp\healthsim-simulation\data\datasets\nhs_dataset_final.csv'
df_combined.to_csv(path_dataset, index=False)

print("✅ Dataset nettoyé et sauvegardé !")
print(f"\nColonnes finales :")
for col in df_combined.columns:
    print(f"  - {col}")
print(f"\nDimensions : {df_combined.shape}")

✅ Dataset nettoyé et sauvegardé !

Colonnes finales :
  - Code_Hopital
  - Nom_Hopital
  - Taux_Mortalite
  - SHMI
  - Nombre_Deces
  - Lits_Disponibles
  - Taux_Occupation
  - Taux_Lits_Vacants
  - Total_Infections_CDI
  - Taux_Infections
  - Duree_Moyenne_Sejour_Jours
  - Taux_Transfert
  - Total_Admissions
  - Temps_Attente_Min
  - Nombre_Medecins
  - Nombre_Infirmiers
  - Ratio_Medecin_Lit
  - Ratio_Infirmier_Lit
  - Ratio_Infirmier_Medecin
  - Total_Equipements
  - Ratio_Equipement_Lit
  - Annee
  - Statut
  - Statut_Label

Dimensions : (206, 24)


In [28]:
# ============================================
# CELLULE — AJOUT COLONNES STATUT PAR KPI
# ============================================

def statut_kpi(valeur, seuil_critique, seuil_attention):
    if valeur > seuil_critique:
        return 'Critique'
    elif valeur > seuil_attention:
        return 'Attention'
    else:
        return 'Normal'

# Statut de chaque KPI
df_combined['Statut_C2'] = df_combined['Taux_Occupation'].apply(
    lambda x: statut_kpi(x, 92, 85))

df_combined['Statut_SHMI'] = df_combined['SHMI'].apply(
    lambda x: statut_kpi(x, 1.15, 1.0))

df_combined['Statut_A2'] = df_combined['Duree_Moyenne_Sejour_Jours'].apply(
    lambda x: statut_kpi(x, 7, 4.5))

df_combined['Statut_A4'] = df_combined['Temps_Attente_Min'].apply(
    lambda x: statut_kpi(x, 720, 240))

df_combined['Statut_A9'] = df_combined['Taux_Transfert'].apply(
    lambda x: statut_kpi(x, 5, 2))

# Sauvegarder
df_combined.to_csv(path_dataset, index=False)

print("✅ Colonnes statut par KPI ajoutées !")
print(f"\nAperçu :")
print(df_combined[['Nom_Hopital', 'Statut_C2', 'Statut_SHMI', 
                    'Statut_A2', 'Statut_A4', 'Statut_A9', 
                    'Statut_Label']].head(10).to_string())

# Sauvegarder avec les nouvelles colonnes
df_combined.to_csv(path_dataset, index=False)
print("✅ Dataset sauvegardé avec colonnes statut KPI !")
print(f"Chemin : {path_dataset}")
print(f"Dimensions : {df_combined.shape}")

✅ Colonnes statut par KPI ajoutées !

Aperçu :
                                                      Nom_Hopital  Statut_C2 Statut_SHMI  Statut_A2  Statut_A4  Statut_A9 Statut_Label
0                                   AIREDALE NHS FOUNDATION TRUST  Attention      Normal     Normal     Normal  Attention    Attention
1           ASHFORD AND ST PETER'S HOSPITALS NHS FOUNDATION TRUST  Attention      Normal     Normal     Normal   Critique    Attention
2  BARKING, HAVERING AND REDBRIDGE UNIVERSITY HOSPITALS NHS TRUST   Critique      Normal     Normal  Attention     Normal    Attention
3                          BARNSLEY HOSPITAL NHS FOUNDATION TRUST   Critique      Normal     Normal     Normal  Attention    Attention
4                                          BARTS HEALTH NHS TRUST   Critique      Normal  Attention     Normal     Normal    Attention
5                     BEDFORDSHIRE HOSPITALS NHS FOUNDATION TRUST   Critique      Normal     Normal     Normal   Critique     Critique
6       

In [29]:
# Voir tout le dataset
pd.set_option('display.max_rows', 206)
pd.set_option('display.max_columns', 30)
df_combined

,Code_Hopital,Nom_Hopital,Taux_Mortalite,SHMI,Nombre_Deces,Lits_Disponibles,Taux_Occupation,Taux_Lits_Vacants,Total_Infections_CDI,Taux_Infections,Duree_Moyenne_Sejour_Jours,Taux_Transfert,Total_Admissions,Temps_Attente_Min,Nombre_Medecins,Nombre_Infirmiers,Ratio_Medecin_Lit,Ratio_Infirmier_Lit,Ratio_Infirmier_Medecin,Total_Equipements,Ratio_Equipement_Lit,Annee,Statut,Statut_Label,Statut_C2,Statut_SHMI,Statut_A2,Statut_A4,Statut_A9
0,RCF,AIREDALE NHS FOUNDATION TRUST,1.632136,0.9745,1095.0,405.197802,86.250102,13.749898,119.0,0.177374,4.027666,2.049486,67090.0,220.580000,338.28359,720.68144,0.834860,1.778592,2.130406,32.0,0.078974,2023-24,1,Attention,Attention,Normal,Normal,Normal,Attention
1,RTK,ASHFORD AND ST PETER'S HOSPITALS NHS FOUNDATIO...,1.526753,0.9874,1635.0,525.835165,86.468412,13.531588,71.0,0.066299,3.144720,6.905407,107090.0,222.160000,660.77034,1171.01243,1.256611,2.226957,1.772193,62.0,0.117908,2023-24,1,Attention,Attention,Normal,Normal,Normal,Critique
2,RF4,"BARKING, HAVERING AND REDBRIDGE UNIVERSITY HOS...",1.566386,0.8984,2645.0,1027.461538,94.765719,5.234281,106.0,0.062774,3.516111,0.793557,168860.0,309.100000,1249.61597,2194.36938,1.216217,2.135719,1.756035,87.0,0.084675,2023-24,1,Attention,Critique,Normal,Normal,Attention,Normal
3,RFF,BARNSLEY HOSPITAL NHS FOUNDATION TRUST,2.069401,0.9660,1640.0,512.637363,92.407288,7.592712,78.0,0.098423,3.644835,2.914826,79250.0,211.120000,424.83765,1037.80683,0.828729,2.024446,2.442832,42.0,0.081929,2023-24,1,Attention,Critique,Normal,Normal,Normal,Attention
4,R1H,BARTS HEALTH NHS TRUST,1.671262,0.9913,3425.0,2072.923077,94.929388,5.070612,175.0,0.085393,5.373930,1.356528,204935.0,231.440000,2960.47284,5037.31642,1.428163,2.430055,1.701524,235.0,0.113366,2023-24,1,Attention,Critique,Normal,Attention,Normal,Normal
5,RC9,BEDFORDSHIRE HOSPITALS NHS FOUNDATION TRUST,1.580976,0.9684,3015.0,1200.611111,93.497756,6.502244,116.0,0.060827,3.398034,6.559870,190705.0,180.340000,1263.02456,2163.72522,1.051985,1.802187,1.713130,101.0,0.084124,2023-24,0,Critique,Critique,Normal,Normal,Normal,Critique
6,RXL,BLACKPOOL TEACHING HOSPITALS NHS FOUNDATION TRUST,2.004938,1.0344,2355.0,777.000000,95.939582,4.060418,124.0,0.105568,5.374062,1.468585,117460.0,328.640000,638.20668,2287.60991,0.821373,2.944157,3.584434,66.0,0.084942,2023-24,1,Attention,Critique,Attention,Attention,Attention,Normal
7,RMC,BOLTON NHS FOUNDATION TRUST,2.113680,1.1497,1705.0,695.725275,89.783766,10.216234,191.0,0.236782,4.061010,7.915453,80665.0,210.880000,591.17191,1717.79153,0.849720,2.469066,2.905739,61.0,0.087678,2023-24,1,Attention,Attention,Attention,Normal,Normal,Critique
8,RAE,BRADFORD TEACHING HOSPITALS NHS FOUNDATION TRUST,1.400389,1.1629,1835.0,705.494505,92.129283,7.870717,48.0,0.036631,3.094164,12.946923,131035.0,215.080000,795.67263,1495.52967,1.127823,2.119832,1.879579,84.0,0.119065,2023-24,0,Critique,Critique,Critique,Normal,Normal,Critique
9,RXQ,BUCKINGHAMSHIRE HEALTHCARE NHS TRUST,1.332233,0.9016,1615.0,567.000000,95.596643,4.403357,78.0,0.064343,3.364101,7.593318,121225.0,301.840000,789.15589,1892.68901,1.391809,3.338076,2.398372,86.0,0.151675,2023-24,0,Critique,Critique,Normal,Normal,Attention,Critique


In [30]:
# ============================================
# CELLULE — CHEMINS NOUVELLES ANNÉES
# ============================================
DATA = r'C:\Users\hp\healthsim-simulation\data'

# ── 2021-22 ──
paths_2122 = {
    'beds'      : rf'{DATA}\Beds-Open-Overnight-Web_File-Q4-2021-22-Final-GHVBN.xlsx',
    'shmi'      : rf'{DATA}\SHMI data files, Apr21-Mar22.zip',
    'cdi'       : rf'{DATA}\Table_6_CDI_march_2022.ods',
    'prov'      : rf'{DATA}\hosp-epis-stat-admi-hosp-prov-2021-22-tab.xlsx',
    'aeqi'      : rf'{DATA}\ECDS_CQI_Open_Data_M12.csv',
    'workforce' : rf'{DATA}\NHS Workforce Statistics, March 2022 England and Organisation.xlsx',
    'nidc'      : None,
}

# ── 2019-20 ──
paths_1920 = {
    'beds'      : rf'{DATA}\1920-Q4-Beds-Open-Overnight-Web_File-Final-DE5WC.xlsx',
    'shmi'      : rf'{DATA}\SHMI data files, Apr19-Mar20.zip',
    'cdi'       : rf'{DATA}\Table_6_CDI_march_2020.ods',
    'prov'      : rf'{DATA}\hosp-epis-stat-admi-prov-2019-20-tab.xlsx',
    'aeqi'      : rf'{DATA}\AE_CQI_M12.xls',
    'workforce' : rf'{DATA}\NHS Workforce Statistics, March 2020 England and Organisation.xlsx',
    'nidc'      : None,
}

# ── 2018-19 ──
paths_1819 = {
    'beds'      : rf'{DATA}\Beds-Open-Overnight-Web_File-Final_Q4_2018-19.xlsx',
    'shmi'      : rf'{DATA}\SHMI data files, Apr18-Mar19.zip',
    'cdi'       : rf'{DATA}\Table_2_Monthly_CDI_2P_April_2018.ods',
    'prov'      : rf'{DATA}\hosp-epis-stat-admi-prov-2018-19-tab.xlsx',
    'aeqi'      : rf'{DATA}\2018-19 M12 AE CQI Suppressed.xlsx',
    'workforce' : rf'{DATA}\NHS Workforce Statistics, March 2019 England and Organisation.xlsx',
    'nidc'      : None,
}

# ── 2017-18 ──
paths_1718 = {
    'beds'      : rf'{DATA}\Beds-Open-Overnight-Q4-201718-REVISED-Web_File-Final.xlsx',
    'shmi'      : rf'{DATA}\SHMI data files, Apr17-Mar18.zip',
    'cdi'       : rf'{DATA}\cdi_table_trust_20190425.ods',
    'prov'      : rf'{DATA}\hosp-epis-stat-admi-prov-2017-18-tab.xlsx',
    'aeqi'      : rf'{DATA}\HESF_AE_CQI_dev_NHSdigital_M12 suppressed v1.1.xlsx',
    'workforce' : rf'{DATA}\NHS Workforce Statistics, March 2018 Organisation - Excel tables.xlsx',
    'nidc'      : None,
}

# ── 2024-25 ──
paths_2425 = {
    'beds'      : rf'{DATA}\Beds-Open-Overnight-Web_File-Q4-2024-25.xlsx',
    'shmi'      : rf'{DATA}\SHMI data, Apr24-Mar25.zip',
    'cdi'       : rf'{DATA}\Table_6_CDI_april_2023_to_march_2024-os.ods',
    'prov'      : rf'{DATA}\hosp-epis-stat-admi-hosp-prov-2024-25-tab.xlsx',
    'aeqi'      : rf'{DATA}\aeqi_open_data_2025_03.csv',
    'workforce' : rf'{DATA}\NHS Workforce Statistics, March 2025 England and Organisation.xlsx',
    'nidc'      : rf'{DATA}\National-Imaging-Data-Collection-Asset-Count-2024-25-v1-FINAL.xlsx',
}

print("✅ Chemins définis !")

✅ Chemins définis !


In [32]:
# ============================================
# CELLULE — CHARGEMENT TOUTES LES ANNÉES
# ============================================

# Chargement des nouvelles années
print("Chargement 2021-22...")
df_2122 = charger_annee('2021-22', paths_2122)

print("\nChargement 2019-20...")
df_1920 = charger_annee('2019-20', paths_1920)

print("\nChargement 2018-19...")
df_1819 = charger_annee('2018-19', paths_1819)

print("\nChargement 2017-18...")
df_1718 = charger_annee('2017-18', paths_1718)

print("\nChargement 2024-25...")
df_2425 = charger_annee('2024-25', paths_2425)

# Combinaison avec les années existantes
df_final = pd.concat([
    df_2324,   # 2023-24 ✅
    df_2223,   # 2022-23 ✅
    df_2122,   # 2021-22
    df_1920,   # 2019-20
    df_1819,   # 2018-19
    df_1718,   # 2017-18
    df_2425,   # 2024-25
], ignore_index=True)

print(f"\n{'='*50}")
print(f"✅ Dataset final : {len(df_final)} trusts × {len(df_final.columns)} colonnes")
print(f"\nDistribution par année :")
print(df_final['annee'].value_counts().sort_index())

Chargement 2021-22...

Chargement année 2021-22...
✅ Beds : 188 trusts
  Fichier SHMI trouvé : SHMI data at trust level, Apr21-Mar22 (csv).csv
✅ SHMI : 121 trusts
✅ CDI : 138 trusts


KeyError: "None of [Index(['Hospital provider code and description†', 'Unnamed: 1', 'Admissions',\n       'Emergency', 'Mean length of stay', 'FCE bed days', 'Emergency.1',\n       'Elective', 'Other'],\n      dtype='str')] are in the [columns]"

In [ ]:
# Voir toutes les colonnes du fichier ERIC
import pandas as pd

DATA = r'C:\Users\hp\healthsim-simulation\data'
df_eric = pd.read_csv(rf'{DATA}\ERIC - 2023_24 - Trust data.csv', nrows=2)

# Afficher toutes les colonnes liées aux équipements
cols_equip = [col for col in df_eric.columns 
              if any(word in col.lower() for word in 
                     ['equip', 'medical', 'maintain', 'operational', 
                      'backlog', 'condemn', 'replac', 'asset'])]

print(f"Total colonnes : {len(df_eric.columns)}")
print(f"\nColonnes liées équipements ({len(cols_equip)}) :")
for col in cols_equip:
    print(f"  - {col}")

print(f"\nToutes les colonnes :")
for col in df_eric.columns:
    print(f"  - {col}")
    

In [ ]:
# ============================================
# CHEMINS — TOUTES LES NOUVELLES ANNÉES
# ============================================
DATA = r'C:\Users\hp\healthsim-simulation\data'

# ── 2012-13 ──
paths_1213 = {
    'beds'      : rf'{DATA}\Beds-Open-Overnight-Web_File-Q4-2012-13-Revised-Final-79659.xls',
    'shmi'      : rf'{DATA}\SHMI data files, Apr12-Mar13.zip',
    'cdi'       : rf'{DATA}\Clostridium_difficile_monthly_data_April_acute_trusts_table_2.xls',
    'prov'      : rf'{DATA}\hosp-epis-stat-admi-hosp-prov-2012-13-tab.xlsx',
    'aeqi'      : rf'{DATA}\prov-ae-qual-indi-eng-mar13-tab1.xls',
    'workforce' : rf'{DATA}\nhs-work-stat-mar-2014-org-tab.xls',
}

# ── 2013-14 ──
paths_1314 = {
    'beds'      : rf'{DATA}\Beds-Open-Overnight-Web_File-Q4-2013-14-Final-Working-Revised-33335.xlsx',
    'shmi'      : rf'{DATA}\SHMI data files, Apr13-Mar14.zip',
    'cdi'       : rf'{DATA}\Clostridium_difficile_monthly_data_April_acute_trusts_table_2.xls',
    'prov'      : rf'{DATA}\hosp-epis-stat-admi-hosp-prov-2013-14-tab.xlsx',
    'aeqi'      : rf'{DATA}\prov-ae-qual-indi-eng-march 2014-tab1.xls',
    'workforce' : rf'{DATA}\nhs-work-stat-mar-2014-org-tab.xls',
}

# ── 2014-15 ──
paths_1415 = {
    'beds'      : rf'{DATA}\Beds-Open-Overnight-Web_File-Q4-2014-15-Revised-Nov15-Final-41885.xlsx',
    'shmi'      : rf'{DATA}\SHMI data files, Apr14-Mar15.zip',
    'cdi'       : rf'{DATA}\Table_2_Monthly_CDI_2P_Mar_2015.xlsx',
    'prov'      : rf'{DATA}\hosp-epis-stat-admi-hosp-prov-2014-15-tab.xlsx',
    'aeqi'      : rf'{DATA}\prov-ae-qual-indi-eng-march 2015-tab1.xls',
    'workforce' : rf'{DATA}\nhs-work-stat-mar-2015-org-tab.xls',
}

# ── 2015-16 ──
paths_1516 = {
    'beds'      : rf'{DATA}\Beds-Open-Overnight-Web_File-Q4-2015-16-Revised-Aug16-Final-84398.xlsx',
    'shmi'      : rf'{DATA}\SHMI data files, Apr15-Mar16.zip',
    'cdi'       : rf'{DATA}\Table_2_Monthly_CDI_2P_March_2016.ods',
    'prov'      : rf'{DATA}\hosp-epis-stat-admi-hosp-prov-2015-16-tab.xlsx',
    'aeqi'      : rf'{DATA}\prov-ae-qual-indi-eng-march 2016-tab1.xls',
    'workforce' : rf'{DATA}\nhs-work-stat-mar-2016-org-tab.xlsx',
}

# ── 2016-17 ──
paths_1617 = {
    'beds'      : rf'{DATA}\Beds-Open-Overnight-Web_File-Q4-2016-17-Final-98123.xlsx',
    'shmi'      : rf'{DATA}\SHMI data files, Apr16-Mar17.zip',
    'cdi'       : rf'{DATA}\Table_2_Monthly_CDI_2P_March_2017.ods',
    'prov'      : rf'{DATA}\hosp-epis-stat-admi-hosp-prov-2016-17-tab.xlsx',
    'aeqi'      : rf'{DATA}\ae_cqi_2016_17_m131.xlsx',
    'workforce' : rf'{DATA}\nhs-work-stat-mar-2017-org-tab.xlsx',
}

# ── 2017-18 ──
paths_1718 = {
    'beds'      : rf'{DATA}\Beds-Open-Overnight-Q4-201718-REVISED-Web_File-Final.xlsx',
    'shmi'      : rf'{DATA}\SHMI data files, Apr17-Mar18.zip',
    'cdi'       : rf'{DATA}\cdi_table_trust_20190425.ods',
    'prov'      : rf'{DATA}\hosp-epis-stat-admi-prov-2017-18-tab.xlsx',
    'aeqi'      : rf'{DATA}\HESF_AE_CQI_dev_NHSdigital_M12 suppressed v1.1.xlsx',
    'workforce' : rf'{DATA}\NHS Workforce Statistics, March 2018 Organisation - Excel tables.xlsx',
}

# ── 2018-19 ──
paths_1819 = {
    'beds'      : rf'{DATA}\Beds-Open-Overnight-Web_File-Final_Q4_2018-19.xlsx',
    'shmi'      : rf'{DATA}\SHMI data files, Apr18-Mar19.zip',
    'cdi'       : rf'{DATA}\Table_2_Monthly_CDI_2P_April_2018.ods',
    'prov'      : rf'{DATA}\hosp-epis-stat-admi-prov-2018-19-tab.xlsx',
    'aeqi'      : rf'{DATA}\2018-19 M12 AE CQI Suppressed.xlsx',
    'workforce' : rf'{DATA}\NHS Workforce Statistics, March 2019 England and Organisation.xlsx',
}

# ── 2019-20 ──
paths_1920 = {
    'beds'      : rf'{DATA}\1920-Q4-Beds-Open-Overnight-Web_File-Final-DE5WC.xlsx',
    'shmi'      : rf'{DATA}\SHMI data files, Apr19-Mar20.zip',
    'cdi'       : rf'{DATA}\Table_6_CDI_march_2020.ods',
    'prov'      : rf'{DATA}\hosp-epis-stat-admi-prov-2019-20-tab.xlsx',
    'aeqi'      : rf'{DATA}\AE_CQI_M12.xls',
    'workforce' : rf'{DATA}\NHS Workforce Statistics, March 2020 England and Organisation.xlsx',
}

# ── 2021-22 ──
paths_2122 = {
    'beds'      : rf'{DATA}\Beds-Open-Overnight-Web_File-Q4-2021-22-Final-GHVBN.xlsx',
    'shmi'      : rf'{DATA}\SHMI data files, Apr21-Mar22.zip',
    'cdi'       : rf'{DATA}\Table_6_CDI_march_2022.ods',
    'prov'      : rf'{DATA}\hosp-epis-stat-admi-hosp-prov-2021-22-tab.xlsx',
    'aeqi'      : rf'{DATA}\ECDS_CQI_Open_Data_M12.csv',
    'workforce' : rf'{DATA}\NHS Workforce Statistics, March 2022 England and Organisation.xlsx',
}

# ── 2024-25 ──
paths_2425 = {
    'beds'      : rf'{DATA}\Beds-Open-Overnight-Web_File-Q4-2024-25.xlsx',
    'shmi'      : rf'{DATA}\SHMI data, Apr24-Mar25.zip',
    'cdi'       : rf'{DATA}\Table_6_CDI_april_2023_to_march_2024-os.ods',
    'prov'      : rf'{DATA}\hosp-epis-stat-admi-hosp-prov-2024-25-tab.xlsx',
    'aeqi'      : rf'{DATA}\aeqi_open_data_2025_03.csv',
    'workforce' : rf'{DATA}\NHS Workforce Statistics, March 2025 England and Organisation.xlsx',
}

print("✅ Chemins définis pour 10 nouvelles années !")

In [ ]:
# ============================================
# TEST — Chargement 2016-17 d'abord
# ============================================
print("Test chargement 2016-17...")
try:
    df_1617 = charger_annee('2016-17', paths_1617)
    print(f"✅ 2016-17 : {len(df_1617)} trusts chargés")
    print(df_1617.head(3))
except Exception as e:
    print(f"❌ Erreur : {e}")

In [36]:
# Voir les colonnes exactes de l'onglet Hospital Providers 2021-22
df_test = pd.read_excel(
    rf'{DATA}\hosp-epis-stat-admi-hosp-prov-2021-22-tab.xlsx',
    sheet_name='Hospital Providers',
    header=None
)
# Afficher les 10 premières lignes pour voir la structure
print(df_test.iloc[:10, :10].to_string())

    0                                                                                                                                                                        1    2   3   4   5   6   7    8    9
0 NaN                                                                                                                                                                      NaN  NaN NaN NaN NaN NaN NaN  NaN  NaN
1 NaN                                                                                                                                                                      NaN  NaN NaN NaN NaN NaN NaN  NaN  NaN
2 NaN                                                                                                                                                                      NaN  NaN NaN NaN NaN NaN NaN  NaN  NaN
3 NaN                                                                                                                                                       Hosp

In [38]:
# Chercher la ligne des headers
for i in range(10, 20):
    print(f"Ligne {i} : {df_test.iloc[i, :10].tolist()}")

Ligne 10 : [nan, 'Hospital provider code and description†', nan, nan, nan, nan, nan, nan, 'Finished consultant episodes', 'Admissions']
Ligne 11 : [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]
Ligne 12 : [nan, 'Total', nan, nan, nan, nan, nan, nan, 19626344, 15979490]
Ligne 13 : [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]
Ligne 14 : [nan, 'Y56', 'LONDON COMMISSIONING REGION', nan, nan, nan, nan, nan, 3002355, 2466175]
Ligne 15 : [nan, 'Y58', 'SOUTH WEST COMMISSIONING REGION', nan, nan, nan, nan, nan, 2012960, 1622950]
Ligne 16 : [nan, 'Y59', 'SOUTH EAST COMMISSIONING REGION', nan, nan, nan, nan, nan, 2787390, 2246695]
Ligne 17 : [nan, 'Y60', 'MIDLANDS COMMISSIONING REGION', nan, nan, nan, nan, nan, 3693755, 3011930]
Ligne 18 : [nan, 'Y61', 'EAST OF ENGLAND COMMISSIONING REGION', nan, nan, nan, nan, nan, 2181730, 1737645]
Ligne 19 : [nan, 'Y62', 'NORTH WEST COMMISSIONING REGION', nan, nan, nan, nan, nan, 2710910, 2222220]


In [40]:
# Voir toutes les colonnes header ligne 10
print("Headers ligne 10 :")
for i, val in enumerate(df_test.iloc[10, :].tolist()):
    if val is not None and str(val) != 'nan':
        print(f"  Col {i} : {val}")

Headers ligne 10 :
  Col 1 : Hospital provider code and description†
  Col 8 : Finished consultant episodes
  Col 9 : Admissions
  Col 10 : Male
  Col 11 : Female
  Col 12 : Gender Unknown
  Col 13 : Emergency
  Col 14 : Waiting list
  Col 15 : Planned
  Col 16 : Other Admission Method
  Col 17 : Mean time waited
  Col 18 : Median time waited
  Col 19 : Mean length of stay
  Col 20 : Median length of stay
  Col 21 : Mean age
  Col 22 : Age 0
  Col 23 : Age 1-4
  Col 24 : Age 5-9
  Col 25 : Age 10-14
  Col 26 : Age 15
  Col 27 : Age 16
  Col 28 : Age 17
  Col 29 : Age 18
  Col 30 : Age 19
  Col 31 : Age 20-24
  Col 32 : Age 25-29
  Col 33 : Age 30-34
  Col 34 : Age 35-39
  Col 35 : Age 40-44
  Col 36 : Age 45-49
  Col 37 : Age 50-54
  Col 38 : Age 55-59
  Col 39 : Age 60-64
  Col 40 : Age 65-69
  Col 41 : Age 70-74
  Col 42 : Age 75-79
  Col 43 : Age 80-84
  Col 44 : Age 85-89
  Col 45 : Age 90+
  Col 46 : Day case
  Col 47 : FCE bed days
  Col 48 : Emergency
  Col 49 : Elective
  Col 5

In [44]:
pip install xlrd --break-system-packages

   ---------------------------------------- 0.0/96.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/96.6 kB ? eta -:--:--
   ---- ----------------------------------- 10.2/96.6 kB ? eta -:--:--
   ---- ----------------------------------- 10.2/96.6 kB ? eta -:--:--
   ------------ --------------------------- 30.7/96.6 kB 259.2 kB/s eta 0:00:01
   ---------------- ----------------------- 41.0/96.6 kB 279.3 kB/s eta 0:00:01
   -------------------------------------- - 92.2/96.6 kB 435.7 kB/s eta 0:00:01
   ---------------------------------------- 96.6/96.6 kB 423.1 kB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [46]:
import pandas as pd
DATA = r'C:\Users\hp\healthsim-simulation\data'

# Voir structure Beds 2012-13
df = pd.read_excel(rf'{DATA}\Beds-Open-Overnight-Web_File-Q4-2012-13-Revised-Final-79659.xls',
                   sheet_name=None)
print("Onglets Beds 2012-13 :")
for s in df.keys():
    print(f"  - {s}")

Onglets Beds 2012-13 :
  - NHS Trust by Sector
  - SHA by Sector
  - Occupied by Specialty
  - Data Quality


In [48]:
df_beds = pd.read_excel(
    rf'{DATA}\Beds-Open-Overnight-Web_File-Q4-2012-13-Revised-Final-79659.xls',
    sheet_name='NHS Trust by Sector',
    header=None
)
print(f"Shape : {df_beds.shape}")
print("\nLignes 0-15 colonnes 0-8 :")
print(df_beds.iloc[:15, :8].to_string())

Shape : (249, 23)

Lignes 0-15 colonnes 0-8 :
     0                    1                                                                                                              2         3         4         5          6                7
0  NaN                  NaN                                                                                                            NaN       NaN       NaN       NaN        NaN              NaN
1  NaN               Title:                                   Average daily number of available and occupied beds open overnight by sector       NaN       NaN       NaN        NaN              NaN
2  NaN             Summary:  KH03 is the collection of data to monitor available and occupied beds open overnight that are consultant led.       NaN       NaN       NaN        NaN              NaN
3  NaN                  NaN                                                                                                            NaN       NaN       NaN       N

In [50]:
print("Headers ligne 14 :")
for i, val in enumerate(df_beds.iloc[14, :].tolist()):
    if str(val) != 'nan':
        print(f"  Col {i} : {val}")

print("\nLignes 15-20 :")
print(df_beds.iloc[15:20, :10].to_string())

Headers ligne 14 :
  Col 1 : Year
  Col 2 : Period
  Col 3 : SHA Code
  Col 4 : Org Code
  Col 5 : Org Name
  Col 6 : Total 
  Col 7 : General & Acute
  Col 8 : Learning Disabilities
  Col 9 : Maternity
  Col 10 : Mental Illness
  Col 12 : Total 
  Col 13 : General & Acute
  Col 14 : Learning Disabilities
  Col 15 : Maternity
  Col 16 : Mental Illness
  Col 18 : Total 
  Col 19 : General & Acute
  Col 20 : Learning Disabilities
  Col 21 : Maternity
  Col 22 : Mental Illness

Lignes 15-20 :
     0        1      2    3    4                                               5              6              7            8            9
15 NaN  2012-13  March  NaN  NaN                                         England  138178.305556  106374.472222  1696.577778  7839.322222
16 NaN      NaN    NaN  NaN  NaN                                             NaN            NaN            NaN          NaN          NaN
17 NaN  2012-13  March  Q30  RE9             SOUTH TYNESIDE NHS FOUNDATION TRUST            41